<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/%D0%A1%D0%BE%D0%B2%D1%80%D0%B5%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5_%D1%8D%D0%BC%D0%B1%D0%B5%D0%B4%D0%B4%D0%B8%D0%BD%D0%B3%D0%B8_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Часть 1. Contrastive Learning для эмбеддингов

## Полная теория с исчерпывающими объяснениями и выводами

---

## 1. Введение: почему contrastive learning стал стандартом

### 1.1 Проблема обучения эмбеддингов

Мы разобрали SBERT, который обучается на парах предложений с известной близостью. Но у SBERT есть ограничения, которые становятся критичными при переходе к реальным задачам.

**Первое ограничение: требуются размеченные пары.** SBERT обучается на SNLI, MultiNLI, STS — датасетах с ручной разметкой. Разметка стоит дорого: чтобы получить 500,000 пар, нужно заплатить аннотаторам, обучить их, проверить качество. Это не масштабируется: для нового домена (медицина, юриспруденция, финансы) нужно снова размечать данные.

**Второе ограничение: бинарные или непрерывные метки.** SBERT использует contrastive loss с бинарными метками (похожи/не похожи) или cosine loss с непрерывными оценками (от 1 до 5). Это ограничивает гибкость: в реальных задачах часто нет явных меток, есть только факт связи (query и документ, который на него отвечает).

**Третье ограничение: нет hard negatives.** SBERT использует случайные negatives, которые могут быть слишком лёгкими. Например, для запроса «как приготовить пасту» случайный negative может быть «прогноз погоды» — это легко различить. Но hard negative «рецепт пиццы» — тоже рецепт, но не паста — требует тонкого различения. Без hard negatives модель не учится этим тонкостям.

**Contrastive learning** решает все три проблемы.

**Первое решение: не нужны метки.** Достаточно знать, что два текста **связаны** (positive pair). Это может быть:

- (query, документ), который на него отвечает;
- (вопрос, ответ);
- (заголовок, текст статьи);
- (перевод на английский, перевод на русский);
- (изображение, подпись к нему).

Такие пары легко получить из данных: логи поисковых систем, пары вопрос-ответ на форумах, параллельные корпуса, клики пользователей.

**Второе решение: in-batch negatives.** В батче из $N$ пар мы получаем $N^2 - N$ отрицательных пар **бесплатно**. Для запроса $q_i$ все документы $d_j$ для $j \neq i$ считаются нерелевантными. Это даёт огромное количество обучающего сигнала без дополнительной разметки.

**Третье решение: hard negatives.** Можно специально искать сложные отрицательные примеры: документы, которые лексически похожи на запрос, но семантически далеки. Это заставляет модель учиться тонким различиям.

Contrastive learning лежит в основе **всех** современных эмбеддингов: E5, BGE, Instructor, GTE, LaBSE, CLIP, SimCLR, MoCo. Понимание contrastive learning необходимо для понимания того, как работают современные системы поиска, рекомендаций и RAG.

### 1.2 Исторический контекст

Идея contrastive learning восходит к работе Хинтона и коллег 2006 года о **contrastive divergence**. Но настоящий бум начался в 2018–2020 годах:

- **2018:** van den Oord et al. предложили **InfoNCE** для обучения представлений.
- **2019:** Hadsell et al. (2006) — contrastive loss для metric learning; Wu et al. — deep metric learning.
- **2020:** SimCLR (Chen et al.) — contrastive learning для изображений; MoCo (He et al.) — momentum contrast.
- **2020:** CLIP (Radford et al.) — contrastive learning для изображений и текста.
- **2021–2022:** E5, BGE, Instructor — contrastive learning для текстовых эмбеддингов.

Сегодня contrastive learning — это **стандарт** для обучения эмбеддингов. Почти все современные модели используют его.

### 1.3 Общая схема

Пусть у нас есть батч из $N$ пар:

$$
\{(q_1, d_1), (q_2, d_2), \ldots, (q_N, d_N)\},
$$

где $q_i$ — запрос (query), $d_i$ — релевантный документ (positive). Каждая пара **связана** (например, вопрос и ответ на него).

**Шаг 1: кодирование.** Все запросы и документы проходят через энкодер (обычно BERT):

$$
u_i = \text{Encoder}(q_i), \quad v_i = \text{Encoder}(d_i).
$$

**Разберём:** энкодер — это BERT или его вариант. Он выдаёт для каждого текста вектор размерности $d_{\text{model}}$. В SBERT используется mean pooling, в E5 и BGE — mean pooling, в некоторых моделях — `[CLS]` или weighted pooling.

**Шаг 2: нормализация.** Нормируем эмбеддинги по L2:

$$
u_i \leftarrow \frac{u_i}{\|u_i\|_2}, \quad v_i \leftarrow \frac{v_i}{\|v_i\|_2}.
$$

**Зачем:** нормализация делает косинусную близость эквивалентной скалярному произведению. Это упрощает вычисления и стабилизирует обучение. После нормализации $\|u_i\| = 1$ и $\|v_i\| = 1$, поэтому $u_i^\top v_j = \cos(u_i, v_j) \in [-1, 1]$.

**Шаг 3: вычисление similarity.** Для всех пар $(i, j)$ вычисляем косинусную близость:

$$
s_{ij} = u_i^\top v_j.
$$

**Разберём:** $s_{ij}$ — это матрица $N \times N$, где элемент $(i, j)$ — similarity между $i$-м запросом и $j$-м документом. Диагональные элементы $s_{ii}$ — это similarity между правильными парами (positive). Внедиагональные $s_{ij}$ для $i \neq j$ — это similarity между неправильными парами (negatives).

**Шаг 4: contrastive loss.** Для каждого $i$:

- **Positive:** $(i, i)$ — релевантный документ.
- **Negatives:** $(i, j)$ для $j \neq i$ — все остальные документы в батче (in-batch negatives).

Loss (InfoNCE):

$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\sum_{j=1}^{N} \exp(s_{ij} / \tau)},
$$

где $\tau$ — temperature.

**Шаг 5: общий loss.**

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_i.
$$

**Интуиция:** для каждого запроса $q_i$ мы хотим, чтобы его релевантный документ $d_i$ имел **максимальную** similarity среди всех документов в батче. Это **мягкая** версия задачи классификации: мы не говорим «это правильный документ, остальные неправильные», а даём вероятность.

---

## 2. InfoNCE Loss: полный вывод

### 2.1 Откуда берётся InfoNCE

InfoNCE (Information Noise Contrastive Estimation) — это функция потерь, которая появилась в работе van den Oord et al. (2018) для обучения представлений. Она основана на идее **различения** правильной пары от шумовых (negative) пар.

**Формально:** пусть у нас есть:

- **Anchor:** $u_i$ (query).
- **Positive:** $v_i$ (релевантный документ).
- **Negatives:** $v_j$ для $j \neq i$ (нерелевантные документы).

Мы хотим обучить модель, которая различает positive от negatives. Это задача **$N$-классовой классификации**: positive — класс 1, negatives — классы 2, ..., $N$.

**Вероятность того, что $v_j$ — правильный документ для $u_i$:**

$$
P(j \mid u_i) = \frac{\exp(s_{ij} / \tau)}{\sum_{k=1}^{N} \exp(s_{ik} / \tau)}.
$$

**Loss:**

$$
\mathcal{L}_i = -\log P(i \mid u_i) = -\log \frac{\exp(s_{ii} / \tau)}{\sum_{k=1}^{N} \exp(s_{ik} / \tau)}.
$$

**Это и есть InfoNCE.**

### 2.2 Связь с взаимной информацией

**InfoNCE** называется так потому, что он связан с **взаимной информацией** между query и document. Формально:

$$
I(u; v) \geq \log N - \mathcal{L}_{\text{InfoNCE}},
$$

где $I(u; v)$ — взаимная информация, $N$ — число negatives. Это означает, что минимизация InfoNCE **максимизирует нижнюю оценку** взаимной информации. Чем больше negatives, тем точнее оценка.

**Практическое следствие:** увеличение $N$ (размера батча) улучшает оценку взаимной информации и, следовательно, качество эмбеддингов. Это объясняет, почему современные модели используют **большие батчи** (256, 512, 1024).

### 2.3 Разбор формулы по частям

**Числитель:** $\exp(s_{ii} / \tau)$.

- $s_{ii} = u_i^\top v_i$ — similarity между query и его positive.
- $\tau$ — temperature.
- Экспонента делает значение положительным и усиливает различия.

**Знаменатель:** $\sum_{k=1}^{N} \exp(s_{ik} / \tau)$.

- Сумма по всем документам в батче, включая positive.
- Это нормировка, которая превращает similarity в распределение вероятностей.

**Loss:** $-\log P(i \mid u_i)$.

- Если модель уверена, что $v_i$ — правильный (P ≈ 1), loss ≈ 0.
- Если модель не уверена (P ≈ 1/N), loss ≈ log N.
- Мы минимизируем loss, что означает максимизацию вероятности positive.

**Пример:** пусть $N = 3$, $s_{ii} / \tau = 10$, $s_{ij} / \tau = 0$ для $j \neq i$. Тогда:

$$
P(i \mid u_i) = \frac{e^{10}}{e^{10} + e^0 + e^0} = \frac{22026}{22028} \approx 0.9999.
$$

$$
\mathcal{L}_i = -\log(0.9999) \approx 0.0001.
$$

Loss почти 0, потому что модель уверена.

**Пример 2:** пусть $s_{ij} / \tau = 0$ для всех $j$. Тогда:

$$
P(i \mid u_i) = \frac{1}{3}.
$$

$$
\mathcal{L}_i = -\log(1/3) = 1.0986.
$$

Loss = log 3, потому что модель не различает positive и negatives.

### 2.4 Temperature $\tau$

**Temperature** $\tau$ управляет «резкостью» распределения softmax.

**Анализ:**

- **$\tau \to 0$:** распределение становится **очень резким**. Одно значение доминирует. Loss фокусируется на hardest negatives.
- **$\tau \to \infty$:** распределение становится **равномерным**. Все negatives имеют одинаковый вес. Loss менее чувствителен.
- **$\tau = 1$:** стандартный softmax.

**Математически:** пусть $s_{ii} = 0.5$, $s_{ij} = 0$ для $j \neq i$. Тогда:

- $\tau = 1$: $P(i) = e^{0.5} / (e^{0.5} + (N-1))$.
- $\tau = 0.1$: $P(i) = e^{5} / (e^{5} + (N-1))$.
- $\tau = 0.01$: $P(i) = e^{50} / (e^{50} + (N-1))$.

При $\tau = 0.01$ $P(i) \approx 1$, потому что $e^{50}$ огромно. Loss почти 0.

**Тонкий момент:** при слишком маленьком $\tau$ градиенты становятся **очень большими** для hard negatives, что может привести к нестабильности. При слишком большом $\tau$ градиенты малы, обучение медленное.

**Обычные значения:** $\tau = 0.05$–$0.1$. Это делает распределение достаточно резким, чтобы фокусироваться на hardest negatives, но не настолько, чтобы обучение было нестабильным.

**Обучаемый temperature:** в некоторых моделях (например, CLIP) $\tau$ — обучаемый параметр. Это позволяет модели адаптировать резкость к данным. В других (SimCLR) — фиксированный.

### 2.5 Градиент InfoNCE

Выведем градиент $\mathcal{L}_i$ по $s_{ij}$.

**Обозначим:**

$$
P_{ij} = \frac{\exp(s_{ij} / \tau)}{\sum_{k=1}^{N} \exp(s_{ik} / \tau)}.
$$

Тогда:

$$
\mathcal{L}_i = -\log P_{ii}.
$$

**Градиент по $s_{ij}$:**

$$
\frac{\partial \mathcal{L}_i}{\partial s_{ij}} = -\frac{1}{P_{ii}} \frac{\partial P_{ii}}{\partial s_{ij}}.
$$

**Случай $j = i$:**

$$
\frac{\partial P_{ii}}{\partial s_{ii}} = \frac{1}{\tau} P_{ii} (1 - P_{ii}).
$$

$$
\frac{\partial \mathcal{L}_i}{\partial s_{ii}} = -\frac{1}{P_{ii}} \cdot \frac{1}{\tau} P_{ii} (1 - P_{ii}) = -\frac{1}{\tau} (1 - P_{ii}).
$$

**Случай $j \neq i$:**

$$
\frac{\partial P_{ii}}{\partial s_{ij}} = -\frac{1}{\tau} P_{ii} P_{ij}.
$$

$$
\frac{\partial \mathcal{L}_i}{\partial s_{ij}} = -\frac{1}{P_{ii}} \cdot \left( -\frac{1}{\tau} P_{ii} P_{ij} \right) = \frac{1}{\tau} P_{ij}.
$$

**Интерпретация:**

- Для positive ($j = i$): градиент $-\frac{1}{\tau} (1 - P_{ii})$. Если $P_{ii} \approx 1$, градиент мал (модель уверена). Если $P_{ii} \approx 0$, градиент велик (модель ошибается).
- Для negatives ($j \neq i$): градиент $\frac{1}{\tau} P_{ij}$. Чем выше вероятность negative, тем сильнее мы его «отталкиваем».

**Тонкий момент:** InfoNCE автоматически фокусируется на **hard negatives** — тех, у которых высокая $P_{ij}$. Это ключевое преимущество перед случайными negatives.

### 2.6 Градиент по эмбеддингам

Теперь выведем градиент по эмбеддингам $u_i$ и $v_j$.

**Градиент по $u_i$:**

$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \sum_{j=1}^{N} \frac{\partial \mathcal{L}_i}{\partial s_{ij}} \frac{\partial s_{ij}}{\partial u_i} = \sum_{j=1}^{N} \frac{\partial \mathcal{L}_i}{\partial s_{ij}} v_j.
$$

Подставляем:

$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = -\frac{1}{\tau} (1 - P_{ii}) v_i + \sum_{j \neq i} \frac{1}{\tau} P_{ij} v_j.
$$

**Перегруппируем:**

$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{ij} v_j - v_i \right).
$$

**Интерпретация:** градиент по $u_i$ — это разность между **взвешенным средним** документов (где веса — это $P_{ij}$) и **правильным документом** $v_i$. Мы сдвигаем $u_i$ в направлении правильного документа и отталкиваем от взвешенного среднего negatives.

**Градиент по $v_j$:**

$$
\frac{\partial \mathcal{L}}{\partial v_j} = \sum_{i=1}^{N} \frac{\partial \mathcal{L}_i}{\partial v_j}.
$$

Для каждого $i$:

$$
\frac{\partial \mathcal{L}_i}{\partial v_j} = \frac{\partial \mathcal{L}_i}{\partial s_{ij}} u_i.
$$

**Случай $j = i$:**

$$
\frac{\partial \mathcal{L}_i}{\partial v_i} = -\frac{1}{\tau} (1 - P_{ii}) u_i.
$$

**Случай $j \neq i$:**

$$
\frac{\partial \mathcal{L}_i}{\partial v_j} = \frac{1}{\tau} P_{ij} u_i.
$$

**Суммируем по всем $i$:**

$$
\frac{\partial \mathcal{L}}{\partial v_j} = \frac{1}{\tau} \left( \sum_{i \neq j} P_{ij} u_i - (1 - P_{jj}) u_j \right).
$$

**Тонкий момент:** градиенты по $u_i$ и $v_j$ зависят от **всех** $P_{ij}$, что делает обучение **глобальным**: каждое обновление учитывает весь батч.

---

## 3. In-Batch Negatives

### 3.1 Идея

**In-batch negatives** — это техника, при которой negatives берутся **из того же батча**. Для батча из $N$ пар:

- Для query $q_i$ positive — это $d_i$.
- Negatives — это все $d_j$ для $j \neq i$.

**Число negatives:** $N - 1$ на каждый query. Общее число пар: $N^2$.

**Преимущество:** мы получаем negatives **бесплатно**. Не нужно специально искать отрицательные примеры.

### 3.2 Эффективность

**Вычислительная сложность:**

- Кодирование: $O(N \cdot d)$ — $N$ запросов и $N$ документов.
- Similarity: $O(N^2 \cdot d)$ — матрица $N \times N$.
- Loss: $O(N^2)$.

**Итого:** $O(N^2 \cdot d)$.

**Пример:** при $N = 256$ это $256^2 = 65536$ пар. Это в 256 раз больше, чем $N$ пар. Это делает обучение **очень эффективным**.

**Тонкий момент:** при $N = 1024$ это $1024^2 \approx 10^6$ пар. Это огромное количество обучающего сигнала. Но требует много памяти: матрица similarity $1024 \times 1024$ занимает $4$ МБ (float32). Это управляемо.

### 3.3 Проблема false negatives

**False negatives** — это пары $(i, j)$, где $j \neq i$, но $d_j$ **тоже релевантен** $q_i$. Например, если в батче два похожих вопроса, их документы могут быть релевантны обоим.

**Пример:** батч содержит:

- $(q_1, d_1)$: «как приготовить пасту» → «рецепт спагетти»
- $(q_2, d_2)$: «как сварить макароны» → «рецепт макарон»

$q_1$ и $q_2$ похожи. $d_1$ и $d_2$ тоже похожи. Для $q_1$ документ $d_2$ — это **false negative**: он релевантен, но мы считаем его negative.

**Проблема:** модель будет «отталкивать» релевантный документ, что ухудшает качество.

**Решения:**

1. **Дедупликация:** удалить дубликаты из батча. Если $q_1$ и $q_2$ очень похожи, оставить только один.
2. **Curriculum learning:** начинать с лёгких negatives, постепенно переходить к сложным.
3. **Cross-batch negatives:** использовать negatives из предыдущих батчей (как в MoCo).
4. **False negative detection:** использовать similarity между $q_i$ и $q_j$ для определения false negatives.

### 3.4 Cross-Batch Negatives

**Cross-batch negatives** — это техника, при которой negatives берутся не только из текущего батча, но и из **предыдущих**. Это увеличивает число negatives без увеличения размера батча.

**Пример:** MoCo использует очередь из $K$ предыдущих эмбеддингов. Для каждого query negatives — это текущий батч + очередь.

**Преимущество:** больше negatives → лучше обучение.

**Недостаток:** сложнее реализация, нужно хранить очередь.

**Формально:** пусть $\mathcal{Q}$ — очередь из $K$ эмбеддингов документов. Для query $u_i$:

$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\exp(s_{ii} / \tau) + \sum_{v_j \in \mathcal{Q}} \exp(u_i^\top v_j / \tau)}.
$$

**Тонкий момент:** очередь обновляется: старые эмбеддинги удаляются, новые добавляются. Это позволяет использовать тысячи negatives без увеличения батча.

---

## 4. Hard Negatives

### 4.1 Идея

**Hard negatives** — это отрицательные примеры, которые **похожи** на query, но **не релевантны**. Например:

- Query: «как приготовить пасту»
- Positive: «рецепт спагетти»
- Easy negative: «прогноз погоды»
- Hard negative: «рецепт пиццы» (тоже рецепт, но не паста)

**Почему hard negatives важны:** если модель обучается только на easy negatives, она учится различать **разные темы**, но не учится различать **близкие темы**. Hard negatives заставляют модель учиться тонким различиям.

**Аналогия:** если вы учите иностранный язык, легко различить «кошка» и «прогноз погоды». Но сложно различить «кошка» и «кошка» (с опечаткой) или «кошка» и «кошак». Hard negatives тренируют именно это тонкое различение.

### 4.2 Как находить hard negatives

**Способ 1: BM25.**

Использовать BM25 для поиска документов, которые лексически похожи на query, но не релевантны.

**Пример:** query «как приготовить паста». BM25 найдёт документы со словами «паста», «приготовить». Среди них могут быть hard negatives (например, «как приготовить пасту для зубов»).

**Преимущество:** BM25 быстрый и не требует обучения.
**Недостаток:** BM25 ловит только лексическую близость, не семантическую.

**Способ 2: другой энкодер.**

Использовать **другую** модель (например, предобученный SBERT) для поиска документов, которые она считает близкими к query. Среди них могут быть hard negatives.

**Преимущество:** ловит семантическую близость.
**Недостаток:** требует другой модели, может быть медленным.

**Способ 3: in-batch mining.**

Использовать текущую модель для поиска hard negatives в батче. Это делается периодически (например, каждые $K$ шагов).

**Преимущество:** hard negatives адаптируются к текущей модели.
**Недостаток:** может быть нестабильным.

### 4.3 Обучение с hard negatives

**Схема:**

1. Для каждого query найти $K$ hard negatives.
2. Обучить модель на парах (query, positive) + (query, hard negatives).

**Loss:**

$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\exp(s_{ii} / \tau) + \sum_{k=1}^{K} \exp(s_{ik}^- / \tau)},
$$

где $s_{ik}^-$ — similarity с hard negative $k$.

**Тонкий момент:** hard negatives должны быть **действительно** нерелевантны. Если они релевантны (false negatives), обучение ухудшается.

### 4.4 Curriculum Learning

**Curriculum learning** — это стратегия, при которой модель сначала обучается на **лёгких** negatives, затем на **сложных**.

**Схема:**

1. **Этап 1:** использовать in-batch negatives (случайные).
2. **Этап 2:** использовать BM25 hard negatives.
3. **Этап 3:** использовать hard negatives из текущей модели.

**Преимущество:** модель постепенно усложняет задачу, что улучшает сходимость.

**Тонкий момент:** E5 и BGE используют curriculum learning. Это одна из ключевых идей их успеха.

---

## 5. Cross-Encoder vs Bi-Encoder vs Late Interaction

### 5.1 Bi-Encoder (SBERT)

**Архитектура:** два предложения кодируются **независимо** одним энкодером.

$$
u = \text{Encoder}(q), \quad v = \text{Encoder}(d).
$$

**Similarity:** косинусная близость $u^\top v$.

**Преимущества:**

- **Быстро:** эмбеддинги можно предвычислить.
- **Масштабируется:** $O(N)$ для $N$ документов.

**Недостатки:**

- **Нет взаимодействия:** query и document не «видят» друг друга при кодировании.

**Применение:** dense retrieval, семантический поиск.

**Тонкий момент:** bi-encoder — это то, что мы разобрали в SBERT. Он используется для **retrieval**: быстро найти кандидатов из миллионов документов.

### 5.2 Cross-Encoder

**Архитектура:** query и document подаются **вместе** в один энкодер.

$$
\text{[CLS]} \quad q \quad \text{[SEP]} \quad d \quad \text{[SEP]}.
$$

**Similarity:** выход `[CLS]` проходит через линейный слой.

**Преимущества:**

- **Точность:** query и document взаимодействуют на всех слоях.

**Недостатки:**

- **Медленно:** нужно $O(N)$ проходов для $N$ документов.
- **Не масштабируется:** нельзя предвычислить.

**Применение:** reranking (переранжирование кандидатов).

**Тонкий момент:** cross-encoder используется для **reranking**: после того как bi-encoder нашёл 100 кандидатов, cross-encoder точно их сортирует.

### 5.3 Late Interaction (ColBERT)

**Архитектура:** query и document кодируются независимо, но similarity вычисляется **поэлементно**.

$$
s(q, d) = \sum_{i \in q} \max_{j \in d} u_i^\top v_j.
$$

**Разберём формулу:**

- $u_i$ — эмбеддинг $i$-го токена query.
- $v_j$ — эмбеддинг $j$-го токена document.
- Для каждого токена query находим **максимальную** similarity с токенами document.
- Суммируем по всем токенам query.

**Преимущества:**

- **Точность:** учитывает взаимодействие на уровне токенов.
- **Скорость:** эмбеддинги можно предвычислить.

**Недостатки:**

- **Больше памяти:** нужно хранить эмбеддинги всех токенов, а не один вектор.

**Применение:** ColBERT, ColBERTv2.

### 5.4 Сравнение

| Свойство | Bi-Encoder | Cross-Encoder | Late Interaction |
|----------|------------|---------------|------------------|
| Скорость | Быстро | Медленно | Средне |
| Точность | Средне | Высоко | Высоко |
| Память | Мало | Мало | Много |
| Предвычисление | Да | Нет | Да |
| Применение | Retrieval | Reranking | Retrieval |

**Тонкий момент:** в реальных системах используется **комбинация**: bi-encoder для retrieval (быстро найти кандидатов), cross-encoder для reranking (точно отсортировать).

---

## 6. Численный пример contrastive learning

### 6.1 Постановка

Пусть у нас есть батч из $N = 3$ пар:

- $(q_1, d_1)$: «как приготовить пасту» → «рецепт спагетти»
- $(q_2, d_2)$: «прогноз погоды» → «завтра будет дождь»
- $(q_3, d_3)$: «как приготовить пиццу» → «рецепт пиццы»

**Параметры:**

- $d = 4$ (размерность эмбеддинга);
- $\tau = 0.1$ (temperature).

### 6.2 Эмбеддинги

Пусть энкодер выдал (для простоты зададим вручную):

**Queries:**

$$
u_1 = (1, 0, 0, 0), \quad u_2 = (0, 1, 0, 0), \quad u_3 = (0, 0, 1, 0).
$$

**Documents:**

$$
v_1 = (0.9, 0.1, 0, 0), \quad v_2 = (0, 0.9, 0.1, 0), \quad v_3 = (0.1, 0.1, 0.9, 0).
$$

**Тонкий момент:** в реальности эмбеддинги не ортогональны. Здесь мы используем ортогональные для наглядности.

### 6.3 Нормализация

Нормируем по L2:

$$
\|u_1\| = 1, \quad \|u_2\| = 1, \quad \|u_3\| = 1.
$$

$$
\|v_1\| = \sqrt{0.81 + 0.01} = \sqrt{0.82} \approx 0.9055.
$$

$$
v_1^{\text{norm}} = (0.9938, 0.1104, 0, 0).
$$

Аналогично:

$$
v_2^{\text{norm}} = (0, 0.9938, 0.1104, 0), \quad v_3^{\text{norm}} = (0.1104, 0.1104, 0.9938, 0).
$$

### 6.4 Similarity matrix

$$
s_{ij} = u_i^\top v_j^{\text{norm}}.
$$

**Строка 1:**

$$
s_{11} = 1 \cdot 0.9938 + 0 + 0 + 0 = 0.9938.
$$

$$
s_{12} = 1 \cdot 0 + 0 + 0 + 0 = 0.
$$

$$
s_{13} = 1 \cdot 0.1104 + 0 + 0 + 0 = 0.1104.
$$

**Строка 2:**

$$
s_{21} = 0, \quad s_{22} = 0.9938, \quad s_{23} = 0.1104.
$$

**Строка 3:**

$$
s_{31} = 0.1104, \quad s_{32} = 0.1104, \quad s_{33} = 0.9938.
$$

**Матрица similarity:**

$$
S = \begin{pmatrix}
0.9938 & 0 & 0.1104 \\
0 & 0.9938 & 0.1104 \\
0.1104 & 0.1104 & 0.9938
\end{pmatrix}.
$$

### 6.5 InfoNCE loss для $i = 1$

$$
\mathcal{L}_1 = -\log \frac{\exp(s_{11} / \tau)}{\sum_{j=1}^{3} \exp(s_{1j} / \tau)}.
$$

$\tau = 0.1$:

$$
s_{11} / \tau = 9.938, \quad s_{12} / \tau = 0, \quad s_{13} / \tau = 1.104.
$$

$$
\exp(9.938) = 20745, \quad \exp(0) = 1, \quad \exp(1.104) = 3.016.
$$

Сумма: $20745 + 1 + 3.016 = 20749$.

$$
P(1 \mid u_1) = 20745 / 20749 = 0.9998.
$$

$$
\mathcal{L}_1 = -\log(0.9998) = 0.0002.
$$

**Аналогично:**

$$
\mathcal{L}_2 = 0.0002, \quad \mathcal{L}_3 = 0.0002.
$$

**Общий loss:**

$$
\mathcal{L} = \frac{1}{3} (0.0002 + 0.0002 + 0.0002) = 0.0002.
$$

**Наблюдение:** loss очень мал, потому что эмбеддинги уже хорошо разделены (positive имеет similarity 0.99, negatives — 0 или 0.11).

### 6.6 Что если эмбеддинги плохие

Пусть энкодер выдал **плохие** эмбеддинги:

$$
u_1 = (0.5, 0.5, 0.5, 0.5), \quad u_2 = (0.5, 0.5, 0.5, 0.5), \quad u_3 = (0.5, 0.5, 0.5, 0.5).
$$

$$
v_1 = (0.5, 0.5, 0.5, 0.5), \quad v_2 = (0.5, 0.5, 0.5, 0.5), \quad v_3 = (0.5, 0.5, 0.5, 0.5).
$$

Тогда все $s_{ij} = 1$. Loss:

$$
\mathcal{L}_1 = -\log \frac{\exp(10)}{\exp(10) + \exp(10) + \exp(10)} = -\log \frac{1}{3} = 1.0986.
$$

**Наблюдение:** loss = 1.0986 = log 3. Это максимальный loss для $N = 3$. Модель не различает positive и negatives.

**Вывод:** contrastive learning заставляет модель **разделять** positive и negatives. Чем лучше разделение, тем меньше loss.

### 6.7 Градиенты для примера

**Градиент по $u_1$:**

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{\tau} \left( \sum_{j=1}^{3} P_{1j} v_j - v_1 \right).
$$

$P_{11} = 0.9998$, $P_{12} = 1/20749 \approx 0.00005$, $P_{13} = 3.016/20749 \approx 0.00015$.

$$
\sum_{j=1}^{3} P_{1j} v_j \approx 0.9998 \cdot (0.9938, 0.1104, 0, 0) + 0.00005 \cdot (0, 0.9938, 0.1104, 0) + 0.00015 \cdot (0.1104, 0.1104, 0.9938, 0).
$$

Первая компонента:

$$
0.9998 \cdot 0.9938 + 0.00005 \cdot 0 + 0.00015 \cdot 0.1104 = 0.9936 + 0.0000166 = 0.9936.
$$

Вторая компонента:

$$
0.9998 \cdot 0.1104 + 0.00005 \cdot 0.9938 + 0.00015 \cdot 0.1104 = 0.1104 + 0.00005 + 0.0000166 = 0.1105.
$$

Третья компонента:

$$
0.9998 \cdot 0 + 0.00005 \cdot 0.1104 + 0.00015 \cdot 0.9938 = 0 + 0.0000055 + 0.000149 = 0.000155.
$$

Четвёртая компонента:

$$
0 + 0 + 0.00015 \cdot 0 = 0.
$$

$$
\sum P_{1j} v_j \approx (0.9936, 0.1105, 0.000155, 0).
$$

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{0.1} \left( (0.9936, 0.1105, 0.000155, 0) - (0.9938, 0.1104, 0, 0) \right).
$$

$$
= 10 \cdot (-0.0002, 0.0001, 0.000155, 0) = (-0.002, 0.001, 0.00155, 0).
$$

**Интерпретация:** градиент очень мал, потому что модель уже уверена. Это означает, что обновление будет маленьким.

**Градиент по $v_1$:**

$$
\frac{\partial \mathcal{L}}{\partial v_1} = \frac{1}{\tau} \left( \sum_{i \neq 1} P_{i1} u_i - (1 - P_{11}) u_1 \right).
$$

$P_{21} = 0$ (потому что $s_{21} = 0$), $P_{31} = 0.00015$.

$$
\sum_{i \neq 1} P_{i1} u_i = 0 \cdot u_2 + 0.00015 \cdot u_3 = (0, 0, 0.00015, 0).
$$

$$
(1 - P_{11}) u_1 = 0.0002 \cdot (1, 0, 0, 0) = (0.0002, 0, 0, 0).
$$

$$
\frac{\partial \mathcal{L}}{\partial v_1} = 10 \cdot ((0, 0, 0.00015, 0) - (0.0002, 0, 0, 0)) = 10 \cdot (-0.0002, 0, 0.00015, 0) = (-0.002, 0, 0.0015, 0).
$$

**Интерпретация:** $v_1$ сдвигается в направлении $u_1$ (потому что $P_{11} \approx 1$) и отталкивается от $u_3$ (потому что $P_{31}$ мала, но не нулевая).

---

## 7. Contrastive Learning в современных моделях

### 7.1 E5

**E5 (EmbEddings from bidirEctional Encoder rEpresentations)** обучается на парах (query, passage) из датасета **CCPairs** (270M пар). Использует:

- **InfoNCE loss** с in-batch negatives.
- **Curriculum learning:** сначала лёгкие negatives, потом hard negatives.
- **Instruction prefix:** для каждой задачи добавляется инструкция (например, «query:», «passage:»).

**Данные:** CCPairs — это пары (title, body) из Common Crawl, отфильтрованные по качеству. 270M пар — это огромный датасет.

**Обучение:** E5 обучается в два этапа:

1. **Этап 1:** обучение на CCPairs с in-batch negatives.
2. **Этап 2:** fine-tuning на конкретных задачах (MS MARCO, NQ, etc.) с hard negatives.

**Результат:** E5 достигает state-of-the-art на BEIR и MTEB.

### 7.2 BGE

**BGE (BAAI General Embedding)** обучается на **multi-stage** схеме:

1. **Этап 1:** обучение на больших парах с in-batch negatives.
2. **Этап 2:** обучение с hard negatives из BM25.
3. **Этап 3:** обучение с hard negatives из текущей модели.

Использует **InfoNCE loss** с temperature.

**Особенность:** BGE использует **retroMAE** — технику предобучения, которая маскирует целые фрагменты текста и восстанавливает их. Это улучшает качество эмбеддингов.

**Результат:** BGE превосходит E5 на многих задачах MTEB.

### 7.3 Instructor

**Instructor** — модель, которая принимает **инструкцию** вместе с текстом:

$$
\text{Encoder}(\text{instruction} + \text{text}).
$$

Например: «Представь этот текст для поиска: ...» или «Классифицируй этот текст: ...».

**Преимущество:** одна модель может решать разные задачи, меняя инструкцию.

**Обучение:** contrastive learning на парах (instruction + query, document).

**Тонкий момент:** Instructor показывает, что инструкция может **адаптировать** эмбеддинги к задаче без fine-tuning.

### 7.4 GTE

**GTE (General Text Embeddings)** — модель от Alibaba. Обучается на **multi-stage** схеме с hard negatives и contrastive loss.

**Особенность:** GTE использует **large batch size** (до 8192) и **cross-batch negatives** для увеличения числа negatives.

**Результат:** GTE достигает state-of-the-art на MTEB.

### 7.5 Общие черты

Все современные модели используют:

1. **InfoNCE loss** (или его варианты).
2. **In-batch negatives.**
3. **Hard negatives.**
4. **Curriculum learning.**
5. **Large batch size** (сотни или тысячи).
6. **Temperature** (обычно 0.02–0.1).

---

## 8. Практические аспекты

### 8.1 Размер батча

**Большой батч** → больше in-batch negatives → лучше обучение.

**Типичные значения:** 256, 512, 1024, 2048.

**Проблема:** большой батч требует много памяти.

**Решение:** gradient accumulation, cross-batch negatives, MoCo-style queue.

**Пример:** если у вас 16 ГБ GPU, вы можете позволить батч 64. Но с gradient accumulation (4 шага) вы получаете эффективный батч 256.

### 8.2 Temperature

**Типичные значения:** 0.02–0.1.

**Как выбирать:** меньше $\tau$ → фокус на hard negatives, но нестабильное обучение. Больше $\tau$ → стабильное обучение, но менее эффективное.

**Обучаемый temperature:** в некоторых моделях $\tau$ — обучаемый параметр. Это позволяет модели адаптировать резкость к данным.

### 8.3 Hard negatives

**Сколько:** обычно 1–7 hard negatives на query.

**Как находить:** BM25, другой энкодер, in-batch mining.

**Тонкий момент:** hard negatives должны быть **действительно** нерелевантны. False negatives ухудшают обучение.

### 8.4 Curriculum

**Типичная схема:**

1. **Этап 1:** in-batch negatives (случайные).
2. **Этап 2:** BM25 hard negatives.
3. **Этап 3:** hard negatives из текущей модели.

**Длительность каждого этапа:** обычно 1–3 эпохи.

### 8.5 Instruction prefix

Многие модели (E5, Instructor) используют **instruction prefix**:

- Для query: «query: » + текст.
- Для document: «passage: » + текст.

**Зачем:** это помогает модели различать роли query и document.

### 8.6 Оценка

**Метрики:**

- **Recall@k:** доля релевантных документов в top-k.
- **MRR (Mean Reciprocal Rank):** средний обратный ранг.
- **NDCG (Normalized Discounted Cumulative Gain):** учитывает порядок.

**Бенчмарки:**

- **MTEB (Massive Text Embedding Benchmark):** 56 датасетов, 112 языков.
- **BEIR:** 18 датасетов для zero-shot retrieval.

**Тонкий момент:** MTEB — это стандарт для оценки эмбеддингов. Он включает задачи retrieval, clustering, classification, reranking, STS.

---

## 9. Математические свойства contrastive learning

### 9.1 Связь с mutual information

InfoNCE — это **нижняя оценка** взаимной информации:

$$
I(u; v) \geq \log N - \mathcal{L}_{\text{InfoNCE}}.
$$

**Доказательство (кратко):** следует из неравенства Йенсена и определения взаимной информации.

**Следствие:** минимизация InfoNCE максимизирует нижнюю оценку $I(u; v)$. Чем больше $N$, тем точнее оценка.

### 9.2 Alignment и uniformity

Contrastive learning оптимизирует два свойства:

1. **Alignment:** positive пары должны быть близки.
2. **Uniformity:** эмбеддинги должны быть равномерно распределены на сфере.

**Alignment:**

$$
\mathcal{L}_{\text{align}} = \mathbb{E}_{(u, v) \sim \text{positive}} \|u - v\|_2^2.
$$

**Uniformity:**

$$
\mathcal{L}_{\text{uniform}} = \log \mathbb{E}_{u, v \sim \text{random}} \exp(-2 \|u - v\|_2^2).
$$

**Тонкий момент:** InfoNCE оптимизирует **оба** свойства одновременно. Это делает его эффективным.

### 9.3 Температура и uniformity

Temperature $\tau$ контролирует **uniformity**. Маленькое $\tau$ → более равномерное распределение. Большое $\tau$ → менее равномерное.

**Следствие:** $\tau$ влияет на качество эмбеддингов. Слишком большое $\tau$ → эмбеддинги «схлопываются». Слишком маленькое $\tau$ → эмбеддинги слишком «разбросаны».

---

## 10. Заключение

Contrastive learning — это фундамент современных эмбеддингов. Ключевые идеи:

1. **InfoNCE loss:** различение positive от negatives.

2. **In-batch negatives:** negatives берутся из батча, что даёт $N^2$ пар бесплатно.

3. **Hard negatives:** сложные отрицательные примеры улучшают обучение.

4. **Temperature:** управляет резкостью распределения.

5. **Curriculum learning:** постепенное усложнение negatives.

6. **Cross-encoder vs bi-encoder vs late interaction:** разные архитектуры для разных задач.

**Ключевые формулы:**

InfoNCE loss:
$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\sum_{j=1}^{N} \exp(s_{ij} / \tau)}.
$$

Градиент:
$$
\frac{\partial \mathcal{L}_i}{\partial s_{ii}} = -\frac{1}{\tau} (1 - P_{ii}), \quad \frac{\partial \mathcal{L}_i}{\partial s_{ij}} = \frac{1}{\tau} P_{ij} \quad (j \neq i).
$$

Градиент по эмбеддингам:
$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{ij} v_j - v_i \right).
$$

Similarity:
$$
s_{ij} = u_i^\top v_j.
$$

**Что дальше:** в следующей части мы разберём конкретные модели E5, BGE, Instructor, GTE — их архитектуру, данные обучения, особенности и результаты на MTEB/BEIR.



# Численный пример Contrastive Learning: полный пошаговый разбор

## 1. Постановка задачи

Рассмотрим задачу обучения эмбеддингов для поиска: даны пары (запрос, документ), где документ релевантен запросу. Мы хотим обучить энкодер так, чтобы релевантные пары имели высокую similarity, а нерелевантные — низкую.

**Батч из $N = 3$ пар:**

| Пара | Query | Document |
|------|-------|----------|
| 1 | «как приготовить пасту» | «рецепт спагетти» |
| 2 | «прогноз погоды» | «завтра будет дождь» |
| 3 | «как приготовить пиццу» | «рецепт пиццы» |

**Параметры:**

- $d = 4$ — размерность эмбеддинга (в реальности 768);
- $\tau = 0.1$ — temperature;
- $\eta = 0.01$ — learning rate;
- $N = 3$ — размер батча.

**Тонкий момент:** в реальных задачах $N = 256$–$1024$, $d = 768$, $\tau = 0.02$–$0.1$. Мы используем маленькие значения, чтобы все вычисления можно было проверить вручную.

---

## 2. Начальные эмбеддинги

### 2.1 Эмбеддинги запросов

Пусть энкодер (например, BERT) выдал для запросов:

$$
u_1 = (1.0, 0.0, 0.0, 0.0) \quad \text{(«как приготовить пасту)},
$$

$$
u_2 = (0.0, 1.0, 0.0, 0.0) \quad \text{(«прогноз погоды)},
$$

$$
u_3 = (0.0, 0.0, 1.0, 0.0) \quad \text{(«как приготовить пиццу)».
$$

**Тонкий момент:** в реальности эмбеддинги не ортогональны. Здесь мы используем ортогональные для наглядности: каждый запрос имеет свою «тему».

### 2.2 Эмбеддинги документов

$$
v_1 = (0.9, 0.1, 0.0, 0.0) \quad \text{(«рецепт спагетти)},
$$

$$
v_2 = (0.0, 0.9, 0.1, 0.0) \quad \text{(«завтра будет дождь)},
$$

$$
v_3 = (0.1, 0.1, 0.9, 0.0) \quad \text{(«рецепт пиццы)».
$$

**Наблюдение:** документы 1 и 3 («рецепт спагетти» и «рецепт пиццы») имеют небольшую общую компоненту (0.1), потому что оба про еду. Это создаёт hard negative: для запроса «как приготовить пасту» документ «рецепт пиццы» — сложный negative.

---

## 3. Нормализация эмбеддингов

### 3.1 Нормализация запросов

$$
\|u_1\|_2 = \sqrt{1.0^2 + 0^2 + 0^2 + 0^2} = 1.0.
$$

$$
u_1^{\text{norm}} = \frac{(1.0, 0, 0, 0)}{1.0} = (1.0, 0, 0, 0).
$$

Аналогично:

$$
u_2^{\text{norm}} = (0, 1.0, 0, 0), \quad u_3^{\text{norm}} = (0, 0, 1.0, 0).
$$

### 3.2 Нормализация документов

$$
\|v_1\|_2 = \sqrt{0.9^2 + 0.1^2 + 0^2 + 0^2} = \sqrt{0.81 + 0.01} = \sqrt{0.82} \approx 0.9055.
$$

$$
v_1^{\text{norm}} = \frac{(0.9, 0.1, 0, 0)}{0.9055} = (0.9938, 0.1104, 0, 0).
$$

$$
\|v_2\|_2 = \sqrt{0^2 + 0.9^2 + 0.1^2 + 0^2} = \sqrt{0.82} \approx 0.9055.
$$

$$
v_2^{\text{norm}} = (0, 0.9938, 0.1104, 0).
$$

$$
\|v_3\|_2 = \sqrt{0.1^2 + 0.1^2 + 0.9^2 + 0^2} = \sqrt{0.01 + 0.01 + 0.81} = \sqrt{0.83} \approx 0.9110.
$$

$$
v_3^{\text{norm}} = \frac{(0.1, 0.1, 0.9, 0)}{0.9110} = (0.1098, 0.1098, 0.9879, 0).
$$

**Тонкий момент:** после нормализации все векторы лежат на единичной сфере. Это означает, что скалярное произведение равно косинусной близости.

---

## 4. Similarity matrix

### 4.1 Вычисление $s_{ij} = u_i^\top v_j$

**Строка 1 ($u_1$ = «как приготовить пасту»):**

$$
s_{11} = u_1^\top v_1^{\text{norm}} = 1.0 \cdot 0.9938 + 0 \cdot 0.1104 + 0 + 0 = 0.9938.
$$

$$
s_{12} = u_1^\top v_2^{\text{norm}} = 1.0 \cdot 0 + 0 \cdot 0.9938 + 0 + 0 = 0.
$$

$$
s_{13} = u_1^\top v_3^{\text{norm}} = 1.0 \cdot 0.1098 + 0 \cdot 0.1098 + 0 \cdot 0.9879 + 0 = 0.1098.
$$

**Строка 2 ($u_2$ = «прогноз погоды»):**

$$
s_{21} = u_2^\top v_1^{\text{norm}} = 0.
$$

$$
s_{22} = u_2^\top v_2^{\text{norm}} = 0.9938.
$$

$$
s_{23} = u_2^\top v_3^{\text{norm}} = 0.1098.
$$

**Строка 3 ($u_3$ = «как приготовить пиццу»):**

$$
s_{31} = u_3^\top v_1^{\text{norm}} = 0.1098.
$$

$$
s_{32} = u_3^\top v_2^{\text{norm}} = 0.1098.
$$

$$
s_{33} = u_3^\top v_3^{\text{norm}} = 0.9879.
$$

### 4.2 Матрица similarity

$$
S = \begin{pmatrix}
0.9938 & 0 & 0.1098 \\
0 & 0.9938 & 0.1098 \\
0.1098 & 0.1098 & 0.9879
\end{pmatrix}.
$$

**Наблюдение:**

- Диагональные элементы (positive): 0.9938, 0.9938, 0.9879 — высокие.
- Внедиагональные элементы (negatives): 0 или 0.1098 — низкие.
- $s_{13} = s_{31} = 0.1098$ — это hard negative: «паста» и «пицца» похожи, но не релевантны.

---

## 5. InfoNCE Loss

### 5.1 Loss для $i = 1$

$$
\mathcal{L}_1 = -\log \frac{\exp(s_{11} / \tau)}{\sum_{j=1}^{3} \exp(s_{1j} / \tau)}.
$$

**Шаг 1: деление на $\tau$.**

$$
s_{11} / \tau = 0.9938 / 0.1 = 9.938.
$$

$$
s_{12} / \tau = 0 / 0.1 = 0.
$$

$$
s_{13} / \tau = 0.1098 / 0.1 = 1.098.
$$

**Шаг 2: экспоненты.**

$$
\exp(9.938) = 20745.5.
$$

$$
\exp(0) = 1.0.
$$

$$
\exp(1.098) = 2.998.
$$

**Шаг 3: сумма.**

$$
Z_1 = 20745.5 + 1.0 + 2.998 = 20749.5.
$$

**Шаг 4: вероятность positive.**

$$
P_{11} = \frac{20745.5}{20749.5} = 0.99981.
$$

**Шаг 5: loss.**

$$
\mathcal{L}_1 = -\log(0.99981) = 0.00019.
$$

### 5.2 Loss для $i = 2$

Аналогично:

$$
\mathcal{L}_2 = 0.00019.
$$

### 5.3 Loss для $i = 3$

$$
s_{31} / \tau = 0.1098 / 0.1 = 1.098, \quad s_{32} / \tau = 1.098, \quad s_{33} / \tau = 0.9879 / 0.1 = 9.879.
$$

$$
\exp(1.098) = 2.998, \quad \exp(1.098) = 2.998, \quad \exp(9.879) = 19530.5.
$$

$$
Z_3 = 2.998 + 2.998 + 19530.5 = 19536.5.
$$

$$
P_{33} = \frac{19530.5}{19536.5} = 0.99969.
$$

$$
\mathcal{L}_3 = -\log(0.99969) = 0.00031.
$$

### 5.4 Общий loss

$$
\mathcal{L} = \frac{1}{3} (\mathcal{L}_1 + \mathcal{L}_2 + \mathcal{L}_3) = \frac{1}{3} (0.00019 + 0.00019 + 0.00031) = 0.00023.
$$

**Наблюдение:** loss очень мал, потому что эмбеддинги уже хорошо разделены. Модель уверена, что positive — правильный.

**Тонкий момент:** в начале обучения эмбеддинги **случайны**, и loss был бы большим ($\approx \log 3 = 1.0986$). Мы начали с «хороших» эмбеддингов, чтобы показать, как выглядит обученная модель.

---

## 6. Плохие эмбеддинги: что если модель не обучена

### 6.1 Случайные эмбеддинги

Пусть энкодер выдал **случайные** эмбеддинги:

$$
u_1 = (0.5, 0.5, 0.5, 0.5), \quad u_2 = (0.5, 0.5, 0.5, 0.5), \quad u_3 = (0.5, 0.5, 0.5, 0.5).
$$

$$
v_1 = (0.5, 0.5, 0.5, 0.5), \quad v_2 = (0.5, 0.5, 0.5, 0.5), \quad v_3 = (0.5, 0.5, 0.5, 0.5).
$$

### 6.2 Нормализация

$$
\|u_i\| = \sqrt{0.25 + 0.25 + 0.25 + 0.25} = 1.0.
$$

$$
u_i^{\text{norm}} = (0.5, 0.5, 0.5, 0.5).
$$

Аналогично для $v_j$.

### 6.3 Similarity matrix

$$
s_{ij} = 0.5 \cdot 0.5 + 0.5 \cdot 0.5 + 0.5 \cdot 0.5 + 0.5 \cdot 0.5 = 1.0 \quad \text{для всех } i, j.
$$

$$
S = \begin{pmatrix}
1.0 & 1.0 & 1.0 \\
1.0 & 1.0 & 1.0 \\
1.0 & 1.0 & 1.0
\end{pmatrix}.
$$

### 6.4 Loss

$$
s_{ij} / \tau = 10 \quad \text{для всех } i, j.
$$

$$
\exp(10) = 22026.5.
$$

$$
Z_i = 3 \cdot 22026.5 = 66079.5.
$$

$$
P_{ii} = \frac{22026.5}{66079.5} = \frac{1}{3}.
$$

$$
\mathcal{L}_i = -\log(1/3) = 1.0986.
$$

$$
\mathcal{L} = 1.0986.
$$

**Наблюдение:** loss = $\log 3 = 1.0986$ — **максимальный** loss для $N = 3$. Модель не различает positive и negatives, потому что все эмбеддинги одинаковы.

**Вывод:** contrastive learning заставляет модель **разделять** positive и negatives. Чем лучше разделение, тем меньше loss.

---

## 7. Градиенты и обновление весов

### 7.1 Градиент по $u_1$

Формула:

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{1j} v_j - v_1 \right).
$$

**Шаг 1: вычисление $P_{1j}$.**

$$
P_{11} = 0.99981, \quad P_{12} = \frac{1.0}{20749.5} = 0.000048, \quad P_{13} = \frac{2.998}{20749.5} = 0.000144.
$$

**Шаг 2: взвешенная сумма.**

$$
\sum_{j=1}^{3} P_{1j} v_j = 0.99981 \cdot v_1 + 0.000048 \cdot v_2 + 0.000144 \cdot v_3.
$$

Подставляем $v_j^{\text{norm}}$:

Первая компонента:

$$
0.99981 \cdot 0.9938 + 0.000048 \cdot 0 + 0.000144 \cdot 0.1098 = 0.99361 + 0 + 0.0000158 = 0.99363.
$$

Вторая компонента:

$$
0.99981 \cdot 0.1104 + 0.000048 \cdot 0.9938 + 0.000144 \cdot 0.1098 = 0.11038 + 0.0000477 + 0.0000158 = 0.11044.
$$

Третья компонента:

$$
0.99981 \cdot 0 + 0.000048 \cdot 0.1104 + 0.000144 \cdot 0.9879 = 0 + 0.0000053 + 0.0001423 = 0.0001476.
$$

Четвёртая компонента:

$$
0 + 0 + 0 = 0.
$$

$$
\sum P_{1j} v_j = (0.99363, 0.11044, 0.0001476, 0).
$$

**Шаг 3: вычитание $v_1$.**

$$
v_1^{\text{norm}} = (0.9938, 0.1104, 0, 0).
$$

$$
\sum P_{1j} v_j - v_1 = (0.99363 - 0.9938, 0.11044 - 0.1104, 0.0001476 - 0, 0 - 0).
$$

$$
= (-0.00017, 0.00004, 0.0001476, 0).
$$

**Шаг 4: деление на $\tau$.**

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{0.1} (-0.00017, 0.00004, 0.0001476, 0) = (-0.0017, 0.0004, 0.001476, 0).
$$

### 7.2 Градиент по $v_1$

Формула:

$$
\frac{\partial \mathcal{L}}{\partial v_1} = \frac{1}{\tau} \left( \sum_{i \neq 1} P_{i1} u_i - (1 - P_{11}) u_1 \right).
$$

**Шаг 1: вычисление $P_{i1}$ для $i \neq 1$.**

$$
P_{21} = \frac{\exp(s_{21} / \tau)}{Z_2} = \frac{\exp(0)}{20749.5} = 0.000048.
$$

$$
P_{31} = \frac{\exp(s_{31} / \tau)}{Z_3} = \frac{2.998}{19536.5} = 0.000153.
$$

**Шаг 2: взвешенная сумма.**

$$
\sum_{i \neq 1} P_{i1} u_i = 0.000048 \cdot u_2 + 0.000153 \cdot u_3.
$$

$$
= 0.000048 \cdot (0, 1, 0, 0) + 0.000153 \cdot (0, 0, 1, 0) = (0, 0.000048, 0.000153, 0).
$$

**Шаг 3: $(1 - P_{11}) u_1$.**

$$
1 - P_{11} = 1 - 0.99981 = 0.00019.
$$

$$
(1 - P_{11}) u_1 = 0.00019 \cdot (1, 0, 0, 0) = (0.00019, 0, 0, 0).
$$

**Шаг 4: вычитание.**

$$
\sum P_{i1} u_i - (1 - P_{11}) u_1 = (0 - 0.00019, 0.000048 - 0, 0.000153 - 0, 0) = (-0.00019, 0.000048, 0.000153, 0).
$$

**Шаг 5: деление на $\tau$.**

$$
\frac{\partial \mathcal{L}}{\partial v_1} = \frac{1}{0.1} (-0.00019, 0.000048, 0.000153, 0) = (-0.0019, 0.00048, 0.00153, 0).
$$

### 7.3 Обновление $u_1$ и $v_1$

$$
u_1 \leftarrow u_1 - \eta \frac{\partial \mathcal{L}_1}{\partial u_1} = (1.0, 0, 0, 0) - 0.01 \cdot (-0.0017, 0.0004, 0.001476, 0).
$$

$$
= (1.0 + 0.000017, 0 - 0.000004, 0 - 0.0000148, 0) = (1.000017, -0.000004, -0.0000148, 0).
$$

**Тонкий момент:** обновление очень маленькое, потому что loss уже мал. Модель почти не меняется.

$$
v_1 \leftarrow v_1 - \eta \frac{\partial \mathcal{L}}{\partial v_1} = (0.9, 0.1, 0, 0) - 0.01 \cdot (-0.0019, 0.00048, 0.00153, 0).
$$

$$
= (0.9 + 0.000019, 0.1 - 0.0000048, 0 - 0.0000153, 0) = (0.900019, 0.099995, -0.0000153, 0).
$$

**Наблюдение:** $v_1$ почти не изменился. Это нормально: модель уже хорошо разделяет positive и negatives.

---

## 8. Обучение с плохих эмбеддингов: как меняется loss

### 8.1 Начальное состояние (случайные эмбеддинги)

Пусть все эмбеддинги равны $(0.5, 0.5, 0.5, 0.5)$. Loss = 1.0986.

### 8.2 Градиент по $u_1$ для плохих эмбеддингов

$$
P_{11} = P_{12} = P_{13} = 1/3.
$$

$$
\sum P_{1j} v_j = \frac{1}{3} v_1 + \frac{1}{3} v_2 + \frac{1}{3} v_3 = \frac{1}{3} (0.5, 0.5, 0.5, 0.5) + \frac{1}{3} (0.5, 0.5, 0.5, 0.5) + \frac{1}{3} (0.5, 0.5, 0.5, 0.5).
$$

$$
= (0.5, 0.5, 0.5, 0.5).
$$

$$
\sum P_{1j} v_j - v_1 = (0.5, 0.5, 0.5, 0.5) - (0.5, 0.5, 0.5, 0.5) = (0, 0, 0, 0).
$$

**Наблюдение:** градиент равен нулю! Это означает, что при одинаковых эмбеддингах модель **не может** учиться. Это называется **collapse** — все эмбеддинги «схлопываются» в одну точку.

**Тонкий момент:** именно поэтому contrastive learning использует **разные** начальные эмбеддинги и **temperature**. Если все эмбеддинги одинаковы, градиент нулевой, и модель не учится.

### 8.3 Что если эмбеддинги разные, но плохие

Пусть:

$$
u_1 = (0.6, 0.4, 0, 0), \quad u_2 = (0, 0.6, 0.4, 0), \quad u_3 = (0, 0, 0.6, 0.4).
$$

$$
v_1 = (0.4, 0.6, 0, 0), \quad v_2 = (0, 0.4, 0.6, 0), \quad v_3 = (0, 0, 0.4, 0.6).
$$

Нормируем:

$$
\|u_1\| = \sqrt{0.36 + 0.16} = \sqrt{0.52} = 0.7211.
$$

$$
u_1^{\text{norm}} = (0.8321, 0.5547, 0, 0).
$$

Аналогично:

$$
u_2^{\text{norm}} = (0, 0.8321, 0.5547, 0), \quad u_3^{\text{norm}} = (0, 0, 0.8321, 0.5547).
$$

$$
v_1^{\text{norm}} = (0.5547, 0.8321, 0, 0), \quad v_2^{\text{norm}} = (0, 0.5547, 0.8321, 0), \quad v_3^{\text{norm}} = (0, 0, 0.5547, 0.8321).
$$

**Similarity matrix:**

$$
s_{11} = 0.8321 \cdot 0.5547 + 0.5547 \cdot 0.8321 = 0.4616 + 0.4616 = 0.9232.
$$

$$
s_{12} = 0.8321 \cdot 0 + 0.5547 \cdot 0.5547 = 0 + 0.3077 = 0.3077.
$$

$$
s_{13} = 0.
$$

$$
s_{21} = 0, \quad s_{22} = 0.9232, \quad s_{23} = 0.3077.
$$

$$
s_{31} = 0, \quad s_{32} = 0.3077, \quad s_{33} = 0.9232.
$$

$$
S = \begin{pmatrix}
0.9232 & 0.3077 & 0 \\
0 & 0.9232 & 0.3077 \\
0 & 0.3077 & 0.9232
\end{pmatrix}.
$$

**Loss для $i = 1$:**

$$
s_{11} / \tau = 9.232, \quad s_{12} / \tau = 3.077, \quad s_{13} / \tau = 0.
$$

$$
\exp(9.232) = 10210.5, \quad \exp(3.077) = 21.69, \quad \exp(0) = 1.
$$

$$
Z_1 = 10210.5 + 21.69 + 1 = 10233.2.
$$

$$
P_{11} = 10210.5 / 10233.2 = 0.99778.
$$

$$
\mathcal{L}_1 = -\log(0.99778) = 0.00222.
$$

**Loss вырос** с 0.00019 до 0.00222, потому что эмбеддинги хуже. Модель будет учиться, чтобы уменьшить loss.

**Градиент по $u_1$:**

$$
P_{11} = 0.99778, \quad P_{12} = 21.69/10233.2 = 0.00212, \quad P_{13} = 1/10233.2 = 0.000098.
$$

$$
\sum P_{1j} v_j = 0.99778 \cdot (0.5547, 0.8321, 0, 0) + 0.00212 \cdot (0, 0.5547, 0.8321, 0) + 0.000098 \cdot (0, 0, 0.5547, 0.8321).
$$

Первая компонента:

$$
0.99778 \cdot 0.5547 + 0 + 0 = 0.5535.
$$

Вторая компонента:

$$
0.99778 \cdot 0.8321 + 0.00212 \cdot 0.5547 + 0 = 0.8303 + 0.00118 = 0.8315.
$$

Третья компонента:

$$
0 + 0.00212 \cdot 0.8321 + 0.000098 \cdot 0.5547 = 0.00176 + 0.000054 = 0.00181.
$$

Четвёртая компонента:

$$
0 + 0 + 0.000098 \cdot 0.8321 = 0.000082.
$$

$$
\sum P_{1j} v_j = (0.5535, 0.8315, 0.00181, 0.000082).
$$

$$
v_1 = (0.5547, 0.8321, 0, 0).
$$

$$
\sum P_{1j} v_j - v_1 = (0.5535 - 0.5547, 0.8315 - 0.8321, 0.00181, 0.000082) = (-0.0012, -0.0006, 0.00181, 0.000082).
$$

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{0.1} (-0.0012, -0.0006, 0.00181, 0.000082) = (-0.012, -0.006, 0.0181, 0.00082).
$$

**Обновление:**

$$
u_1 \leftarrow (0.6, 0.4, 0, 0) - 0.01 \cdot (-0.012, -0.006, 0.0181, 0.00082).
$$

$$
= (0.6 + 0.00012, 0.4 + 0.00006, 0 - 0.000181, 0 - 0.0000082) = (0.60012, 0.40006, -0.000181, -0.0000082).
$$

**Наблюдение:** $u_1$ сдвигается в направлении $v_1$ (первая компонента растёт, вторая растёт), и немного отталкивается от $v_2$ (третья компонента уменьшается). Это именно то, что нужно: positive притягивается, negatives отталкиваются.

---

## 9. Обучение с hard negatives

### 9.1 Постановка

Пусть у нас есть **hard negative** для запроса «как приготовить пасту»: документ «рецепт пиццы» ($v_3$). Это hard negative, потому что оба про еду.

**Similarity:**

$$
s_{11} = 0.9938 \quad \text{(positive: «рецепт спагетти»)}.
$$

$$
s_{13} = 0.1098 \quad \text{(hard negative: «рецепт пиццы»)}.
$$

### 9.2 Loss с hard negatives

$$
\mathcal{L}_1 = -\log \frac{\exp(s_{11} / \tau)}{\exp(s_{11} / \tau) + \exp(s_{12} / \tau) + \exp(s_{13} / \tau)}.
$$

Мы уже вычислили: $\mathcal{L}_1 = 0.00019$.

**Тонкий момент:** hard negative $s_{13} = 0.1098$ даёт вклад $\exp(1.098) = 2.998$ в знаменатель. Это больше, чем easy negative $s_{12} = 0$ (вклад 1.0). Поэтому hard negative **сильнее** влияет на loss.

### 9.3 Что если hard negative слишком близок

Пусть $s_{13} = 0.5$ (hard negative очень похож на query). Тогда:

$$
s_{13} / \tau = 5.0, \quad \exp(5.0) = 148.4.
$$

$$
Z_1 = 20745.5 + 1.0 + 148.4 = 20894.9.
$$

$$
P_{11} = 20745.5 / 20894.9 = 0.9929.
$$

$$
\mathcal{L}_1 = -\log(0.9929) = 0.0071.
$$

**Loss вырос** с 0.00019 до 0.0071. Модель будет сильнее обновляться, чтобы **оттолкнуть** hard negative.

**Градиент по $u_1$:**

$$
P_{11} = 0.9929, \quad P_{12} = 1/20894.9 = 0.000048, \quad P_{13} = 148.4/20894.9 = 0.0071.
$$

$$
\sum P_{1j} v_j = 0.9929 \cdot v_1 + 0.000048 \cdot v_2 + 0.0071 \cdot v_3.
$$

Первая компонента:

$$
0.9929 \cdot 0.9938 + 0 + 0.0071 \cdot 0.1098 = 0.9867 + 0.00078 = 0.9875.
$$

Вторая компонента:

$$
0.9929 \cdot 0.1104 + 0.000048 \cdot 0.9938 + 0.0071 \cdot 0.1098 = 0.1096 + 0.000048 + 0.00078 = 0.1104.
$$

Третья компонента:

$$
0 + 0.000048 \cdot 0.1104 + 0.0071 \cdot 0.9879 = 0.0000053 + 0.00701 = 0.00702.
$$

$$
\sum P_{1j} v_j - v_1 = (0.9875 - 0.9938, 0.1104 - 0.1104, 0.00702 - 0, 0) = (-0.0063, 0, 0.00702, 0).
$$

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{0.1} (-0.0063, 0, 0.00702, 0) = (-0.063, 0, 0.0702, 0).
$$

**Наблюдение:** градиент по третьей компоненте $0.0702$ — большой. Это означает, что $u_1$ будет **сильно** отталкиваться от $v_3$ (hard negative). Третья компонента $u_1$ уменьшится.

**Обновление:**

$$
u_1 \leftarrow (1.0, 0, 0, 0) - 0.01 \cdot (-0.063, 0, 0.0702, 0) = (1.00063, 0, -0.000702, 0).
$$

**Наблюдение:** $u_1$ сдвинулся в направлении $v_1$ (первая компонента) и **оттолкнулся** от $v_3$ (третья компонента стала отрицательной). Это именно то, что нужно.

---

## 10. Сводка всех шагов

| Шаг | Компонент | Что происходит |
|-----|-----------|----------------|
| 1 | Эмбеддинги | Запросы и документы → векторы |
| 2 | Нормализация | L2-нормализация |
| 3 | Similarity | $s_{ij} = u_i^\top v_j$ |
| 4 | Temperature | Деление на $\tau$ |
| 5 | Softmax | Вероятности $P_{ij}$ |
| 6 | InfoNCE loss | $-\log P_{ii}$ |
| 7 | Градиент | $\frac{1}{\tau} (\sum P_{ij} v_j - v_i)$ |
| 8 | Обновление | $u_i \leftarrow u_i - \eta \nabla$ |

---

## 11. Что было упрощено

| Компонент | Реальный contrastive learning | Наш пример |
|-----------|-------------------------------|------------|
| $N$ (батч) | 256–1024 | 3 |
| $d$ (размерность) | 768 | 4 |
| $\tau$ | 0.02–0.1 | 0.1 |
| Энкодер | BERT | Заданные векторы |
| Данные | 270M пар | 3 пары |
| Hard negatives | BM25, модель | Заданные |
| Curriculum | 3 этапа | 1 этап |

**Тем не менее** пример показывает **все ключевые шаги** contrastive learning:

1. Кодирование запросов и документов.
2. L2-нормализация.
3. Similarity matrix.
4. Temperature.
5. Softmax.
6. InfoNCE loss.
7. Градиенты.
8. Обновление весов.
9. Hard negatives.

---

## 12. Заключение

В этом численном примере мы шаг за шагом вычислили contrastive learning для батча из трёх пар. Основные выводы:

1. **InfoNCE loss** заставляет модель различать positive от negatives. Loss = 0.00023 для хороших эмбеддингов и 1.0986 для плохих.

2. **Temperature** $\tau = 0.1$ делает распределение резким, фокусируясь на hardest negatives.

3. **In-batch negatives** дают $N^2 = 9$ пар бесплатно.

4. **Градиент** по $u_i$ — это разность между взвешенным средним документов и правильным документом. Модель притягивает positive и отталкивает negatives.

5. **Hard negatives** (например, «рецепт пиццы» для запроса «как приготовить пасту») дают больший градиент, чем easy negatives.

6. **Обновление весов** маленькое, потому что loss уже мал. При плохих эмбеддингах обновление больше.

**Ключевые формулы:**

InfoNCE loss:
$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\sum_{j=1}^{N} \exp(s_{ij} / \tau)}.
$$

Градиент по $u_i$:
$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{ij} v_j - v_i \right).
$$

Similarity:
$$
s_{ij} = u_i^\top v_j.
$$

Этот пример, несмотря на маленькие размерности, даёт полное понимание того, как работает contrastive learning. В реальных моделях $N = 256$–$1024$, $d = 768$, но математика остаётся той же.


# Часть 2. Современные модели эмбеддингов: E5, BGE, Instructor, GTE

## Полная теория с формулами и объяснениями

---

## 1. Введение: почему появились современные модели

### 1.1 Ограничения SBERT

SBERT, как мы разобрали, стал важным шагом вперёд: он показал, что BERT можно адаптировать для генерации эмбеддингов предложений. Но у SBERT есть ограничения, которые стали очевидны при переходе к реальным задачам поиска.

**Первое ограничение: маленькие датасеты.** SBERT обучается на SNLI (570K пар) + MultiNLI (430K пар) + STS (несколько тысяч пар). Это $\approx 1M$ пар. Для сравнения: современные модели обучаются на **сотнях миллионов** пар. Это критично: больше данных → лучше качество.

**Второе ограничение: нет hard negatives.** SBERT использует случайные negatives из батча. Это easy negatives, которые модель быстро учится различать. Но реальные задачи требуют различения **близких** документов. Без hard negatives модель не учится тонким различиям.

**Третье ограничение: нет curriculum learning.** SBERT обучается за один этап. Современные модели используют multi-stage обучение: сначала на больших данных с in-batch negatives, затем на hard negatives, затем fine-tuning.

**Четвёртое ограничение: нет instruction prefix.** SBERT не различает роли query и document. Современные модели (E5, Instructor) добавляют инструкцию, чтобы модель знала, что она обрабатывает: запрос или документ.

### 1.2 Что изменилось

Современные модели (E5, BGE, Instructor, GTE) решают эти ограничения:

1. **Огромные датасеты:** 270M пар (E5), 1B+ пар (BGE), 300M+ пар (GTE).
2. **Hard negatives:** BM25, cross-encoder, in-batch mining.
3. **Multi-stage обучение:** 2–3 этапа с постепенным усложнением.
4. **Instruction prefix:** «query: », «passage: », «Represent this for retrieval: ».
5. **Большие батчи:** 256–8192.
6. **Contrastive loss** с temperature.

В этой части мы разберём каждую модель подробно: архитектуру, данные, обучение, формулы, результаты.

---

## 2. E5: EmbEddings from bidirEctional Encoder rEpresentations

### 2.1 Общая идея

**E5** — это семейство моделей для текстовых эмбеддингов, предложенное Microsoft Research в 2022 году. Название расшифровывается как **EmbEddings from bidirEctional Encoder rEpresentations** — «эмбеддинги из двунаправленных представлений энкодера».

**Ключевая идея:** обучить BERT-подобный энкодер на **огромном** датасете пар (query, passage), используя contrastive learning с in-batch negatives и curriculum learning.

**Особенность:** E5 использует **instruction prefix** — специальные префиксы, которые говорят модели, что она обрабатывает:

- `query: ` — для запросов;
- `passage: ` — для документов.

**Пример:**

- Query: `query: как приготовить пасту`
- Passage: `passage: рецепт спагетти`

Это помогает модели различать роли: эмбеддинги запросов и документов находятся в **разных** подпространствах, но сопоставимы.

### 2.2 Архитектура E5

E5 использует **стандартный BERT encoder** с mean pooling:

**Шаг 1: токенизация.**

$$
\text{query: как приготовить пасту} \to [\text{[CLS]}, \text{query}, \text{:}, \text{как}, \ldots, \text{[SEP]}].
$$

**Шаг 2: BERT encoder.**

$$
h_i = \text{BERT}(x_1, \ldots, x_n)_i.
$$

**Шаг 3: mean pooling.**

$$
u = \frac{1}{n} \sum_{i=1}^{n} h_i.
$$

**Шаг 4: нормализация.**

$$
u \leftarrow \frac{u}{\|u\|_2}.
$$

**Размерности:**

- $d_{\text{model}} = 768$ (E5-base) или $1024$ (E5-large);
- $N = 12$ слоёв (E5-base) или $24$ (E5-large);
- $h = 12$ голов (E5-base) или $16$ (E5-large);
- выход: вектор размерности 768 или 1024.

**Тонкий момент:** E5 использует **mean pooling**, а не `[CLS]`. Эксперименты показывают, что mean pooling лучше для similarity.

### 2.3 Данные E5: CCPairs

**CCPairs** (Colossal Clean Pairs) — это датасет из **270 миллионов** пар (title, body) из Common Crawl. Это огромный корпус, который включает:

- заголовки статей и их текст;
- вопросы и ответы на форумах;
- пары (query, document) из поисковых логов.

**Фильтрация:**

1. Удаление дубликатов.
2. Удаление пар с низким качеством (короткие, шумные).
3. Фильтрация по языку.

**Размер:** 270M пар. Это в 270 раз больше, чем SNLI+MultiNLI, на которых обучается SBERT.

**Тонкий момент:** CCPairs не размечен вручную. Пары (title, body) считаются релевантными, потому что title описывает body. Это **weak supervision** — слабая разметка, которая не требует аннотаторов.

### 2.4 Обучение E5

E5 обучается в **два этапа**.

**Этап 1: contrastive pre-training на CCPairs.**

- Данные: 270M пар.
- Loss: InfoNCE с in-batch negatives.
- Batch size: 32,768 (очень большой).
- Temperature: $\tau = 0.01$.
- Эпохи: 1–2.
- Обучение: несколько дней на TPU.

**Формула loss:**

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j=1}^{N} \exp(u_i^\top v_j / \tau)}.
$$

где:

- $u_i = \text{Encoder}(\text{query: } q_i)$;
- $v_i = \text{Encoder}(\text{passage: } d_i)$;
- $N = 32,768$ — размер батча.

**Тонкий момент:** при $N = 32768$ in-batch negatives дают $N^2 \approx 10^9$ пар. Это огромное количество обучающего сигнала.

**Этап 2: fine-tuning на конкретных задачах.**

- Данные: MS MARCO, NQ, TriviaQA, etc.
- Loss: InfoNCE + hard negatives.
- Hard negatives: из BM25 + cross-encoder.
- Batch size: 256–1024.

**Curriculum learning:**

1. Сначала easy negatives (случайные из батча).
2. Затем hard negatives из BM25.
3. Затем hard negatives из cross-encoder.

### 2.5 Instruction prefix в E5

E5 использует **разные** префиксы для разных задач:

| Задача | Query prefix | Passage prefix |
|--------|--------------|----------------|
| Retrieval | `query: ` | `passage: ` |
| STS | `query: ` | `query: ` |
| Classification | `query: ` | — |
| Clustering | `query: ` | — |

**Пример:**

- Query: `query: как приготовить пасту`
- Passage: `passage: рецепт спагетти`

**Зачем:** префикс помогает модели понять роль текста. Без префикса модель не знает, что `как приготовить пасту` — это запрос, а `рецепт спагетти` — документ.

**Тонкий момент:** префиксы **обучаются** вместе с моделью. Это часть input, а не отдельный параметр.

### 2.6 Формулы E5

**Кодирование:**

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(\text{query: } q)_i \right).
$$

$$
v = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(\text{passage: } d)_i \right).
$$

**Similarity:**

$$
s(q, d) = u^\top v.
$$

**InfoNCE loss:**

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j=1}^{N} \exp(u_i^\top v_j / \tau)}.
$$

**Градиент:**

$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{ij} v_j - v_i \right).
$$

где $P_{ij} = \text{softmax}(u_i^\top v_j / \tau)$.

### 2.7 Результаты E5

**MTEB (Massive Text Embedding Benchmark):**

| Модель | Средний балл | Retrieval | STS | Classification |
|--------|--------------|-----------|-----|----------------|
| SBERT | 52.0 | 40.0 | 75.0 | 60.0 |
| E5-base | 60.0 | 48.0 | 80.0 | 68.0 |
| E5-large | 62.0 | 50.0 | 82.0 | 70.0 |

**BEIR (zero-shot retrieval):**

| Модель | nDCG@10 |
|--------|---------|
| BM25 | 40.0 |
| SBERT | 42.0 |
| E5-base | 47.0 |
| E5-large | 49.0 |

**Наблюдение:** E5 значительно превосходит SBERT на retrieval и BEIR.

---

## 3. BGE: BAAI General Embedding

### 3.1 Общая идея

**BGE** (BAAI General Embedding) — семейство моделей от Пекинской академии искусственного интеллекта (BAAI), предложенное в 2023 году. BGE достигает state-of-the-art на MTEB и BEIR.

**Ключевые идеи:**

1. **Multi-stage обучение:** три этапа с постепенным усложнением.
2. **RetroMAE:** техника предобучения, которая маскирует целые фрагменты текста.
3. **Hard negatives:** BM25 + cross-encoder + in-batch mining.
4. **Большие батчи:** до 8192.

### 3.2 Архитектура BGE

BGE использует **стандартный BERT encoder** с mean pooling:

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(x_1, \ldots, x_n)_i \right).
$$

**Размерности:**

- BGE-base: $d_{\text{model}} = 768$, $N = 12$;
- BGE-large: $d_{\text{model}} = 1024$, $N = 24$;
- BGE-small: $d_{\text{model}} = 384$, $N = 6$.

**Тонкий момент:** BGE не использует instruction prefix, в отличие от E5. Вместо этого BGE использует **разные** модели для разных языков и задач.

### 3.3 RetroMAE

**RetroMAE** — это техника предобучения, которая улучшает качество эмбеддингов. Идея:

1. **Encoder:** текст маскируется (например, 30% токенов), encoder обрабатывает замаскированный текст.
2. **Decoder:** восстанавливает оригинальный текст по эмбеддингу.
3. **Loss:** комбинация reconstruction loss и contrastive loss.

**Формально:**

$$
\mathcal{L}_{\text{RetroMAE}} = \mathcal{L}_{\text{reconstruct}} + \lambda \mathcal{L}_{\text{contrastive}}.
$$

**Reconstruction loss:**

$$
\mathcal{L}_{\text{reconstruct}} = -\sum_{i \in \mathcal{M}} \log P(w_i \mid \text{Encoder}(\tilde{x})),
$$

где $\mathcal{M}$ — множество замаскированных позиций, $\tilde{x}$ — замаскированный вход.

**Contrastive loss:**

$$
\mathcal{L}_{\text{contrastive}} = -\log \frac{\exp(u^\top v / \tau)}{\sum_{j} \exp(u^\top v_j / \tau)}.
$$

**Зачем RetroMAE:** reconstruction loss заставляет encoder **сжимать** информацию в эмбеддинг, потому что decoder должен восстановить весь текст по одному вектору. Это улучшает качество эмбеддингов.

**Тонкий момент:** RetroMAE — это **предобучение**, а не fine-tuning. BGE сначала предобучается с RetroMAE, затем обучается contrastive learning.

### 3.4 Multi-stage обучение BGE

BGE обучается в **три этапа**.

**Этап 1: contrastive pre-training на больших данных.**

- Данные: 1B+ пар (Wudao, Wikipedia, etc.).
- Loss: InfoNCE с in-batch negatives.
- Batch size: 8192.
- Temperature: $\tau = 0.01$.
- Эпохи: 1–2.

**Этап 2: обучение с hard negatives из BM25.**

- Данные: те же пары + hard negatives из BM25.
- Loss: InfoNCE + hard negatives.
- Batch size: 1024.
- Hard negatives: 1–7 на query.

**Формула loss:**

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\exp(u_i^\top v_i / \tau) + \sum_{k=1}^{K} \exp(u_i^\top v_k^- / \tau) + \sum_{j \neq i} \exp(u_i^\top v_j / \tau)}.
$$

где $v_k^-$ — hard negatives, $v_j$ — in-batch negatives.

**Этап 3: обучение с hard negatives из cross-encoder.**

- Данные: те же + hard negatives из cross-encoder.
- Loss: InfoNCE + hard negatives.
- Batch size: 256.
- Hard negatives: 1–3 на query.

**Тонкий момент:** cross-encoder hard negatives сложнее, чем BM25. Они требуют больше вычислений, но дают лучшее качество.

### 3.5 Формулы BGE

**Кодирование:**

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(q)_i \right).
$$

$$
v = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(d)_i \right).
$$

**InfoNCE loss с hard negatives:**

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\exp(u_i^\top v_i / \tau) + \sum_{k=1}^{K} \exp(u_i^\top v_k^- / \tau) + \sum_{j \neq i} \exp(u_i^\top v_j / \tau)}.
$$

**Градиент:**

$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( P_{ii} v_i + \sum_{k} P_{ik}^- v_k^- + \sum_{j \neq i} P_{ij} v_j - v_i \right).
$$

### 3.6 Результаты BGE

**MTEB:**

| Модель | Средний балл | Retrieval | STS | Classification |
|--------|--------------|-----------|-----|----------------|
| E5-large | 62.0 | 50.0 | 82.0 | 70.0 |
| BGE-large | 64.0 | 52.0 | 83.0 | 72.0 |
| BGE-base | 63.0 | 51.0 | 82.0 | 71.0 |

**BEIR:**

| Модель | nDCG@10 |
|--------|---------|
| E5-large | 49.0 |
| BGE-large | 52.0 |
| BGE-base | 51.0 |

**Наблюдение:** BGE превосходит E5 на MTEB и BEIR.

### 3.7 BGE-M3

**BGE-M3** — это расширение BGE, которое поддерживает:

1. **Multi-linguality:** 100+ языков.
2. **Multi-functionality:** dense, sparse, и multi-vector retrieval.
3. **Multi-granularity:** от коротких предложений до длинных документов (до 8192 токенов).

**Тонкий момент:** BGE-M3 использует **три** типа эмбеддингов:

- **Dense:** стандартный вектор (как в BGE).
- **Sparse:** разреженный вектор (как в BM25).
- **Multi-vector:** эмбеддинги для каждого токена (как в ColBERT).

Это позволяет комбинировать разные типы retrieval.

---

## 4. Instructor: Instruction-based Embeddings

### 4.1 Общая идея

**Instructor** — модель, которая принимает **инструкцию** вместе с текстом:

$$
\text{Encoder}(\text{instruction} + \text{text}).
$$

Например: «Represent this text for retrieval: ...» или «Classify this text: ...».

**Ключевая идея:** одна модель может решать **разные** задачи, меняя инструкцию. Это позволяет адаптировать эмбеддинги к задаче **без** fine-tuning.

### 4.2 Архитектура Instructor

Instructor использует **стандартный BERT encoder** с mean pooling, но вход включает инструкцию:

$$
\text{input} = \text{instruction} + \text{text}.
$$

**Пример:**

- Для retrieval: `Represent the query for retrieving supporting documents: как приготовить пасту`
- Для classification: `Classify the text into positive or negative: этот фильм был ужасен`
- Для clustering: `Identify the topic of the text: кошка сидит на окне`

**Формально:**

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(\text{instruction} \oplus \text{text})_i \right).
$$

где $\oplus$ — конкатенация.

### 4.3 Обучение Instructor

Instructor обучается на **нескольких** датасетах одновременно:

1. **Retrieval:** MS MARCO, NQ, etc.
2. **STS:** STS benchmark.
3. **Classification:** SNLI, MultiNLI, etc.
4. **Clustering:** Reddit, etc.

**Loss:** комбинация contrastive loss и classification loss.

**Contrastive loss:**

$$
\mathcal{L}_{\text{contrastive}} = -\log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j} \exp(u_i^\top v_j / \tau)}.
$$

**Classification loss:**

$$
\mathcal{L}_{\text{classification}} = -\log P(y \mid u).
$$

**Общий loss:**

$$
\mathcal{L} = \mathcal{L}_{\text{contrastive}} + \lambda \mathcal{L}_{\text{classification}}.
$$

### 4.4 Instruction prefix в Instructor

Instructor использует **разные** инструкции для разных задач:

| Задача | Инструкция |
|--------|------------|
| Retrieval (query) | `Represent the query for retrieving supporting documents:` |
| Retrieval (document) | `Represent the document for retrieval:` |
| STS | `Represent the sentence for similarity:` |
| Classification | `Classify the text:` |
| Clustering | `Identify the topic of the text:` |

**Тонкий момент:** инструкции **обучаются** вместе с моделью. Это часть input, а не отдельный параметр.

### 4.5 Формулы Instructor

**Кодирование:**

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(\text{instruction} \oplus \text{text})_i \right).
$$

**Similarity:**

$$
s(q, d) = u_q^\top u_d.
$$

**InfoNCE loss:**

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j=1}^{N} \exp(u_i^\top v_j / \tau)}.
$$

### 4.6 Результаты Instructor

**MTEB:**

| Модель | Средний балл | Retrieval | STS | Classification |
|--------|--------------|-----------|-----|----------------|
| BGE-large | 64.0 | 52.0 | 83.0 | 72.0 |
| Instructor-large | 63.0 | 51.0 | 82.0 | 73.0 |
| Instructor-xl | 64.0 | 52.0 | 83.0 | 74.0 |

**Наблюдение:** Instructor сравним с BGE, но лучше на classification, потому что использует инструкции.

**Преимущество Instructor:** одна модель может решать разные задачи, меняя инструкцию. Это удобно для production.

---

## 5. GTE: General Text Embeddings

### 5.1 Общая идея

**GTE** (General Text Embeddings) — семейство моделей от Alibaba, предложенное в 2023 году. GTE достигает state-of-the-art на MTEB.

**Ключевые идеи:**

1. **Multi-stage обучение:** три этапа.
2. **Большие батчи:** до 8192.
3. **Cross-batch negatives:** negatives из предыдущих батчей.
4. **Hard negatives:** BM25 + cross-encoder.

### 5.2 Архитектура GTE

GTE использует **стандартный BERT encoder** с mean pooling:

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(x_1, \ldots, x_n)_i \right).
$$

**Размерности:**

- GTE-base: $d_{\text{model}} = 768$, $N = 12$;
- GTE-large: $d_{\text{model}} = 1024$, $N = 24$;
- GTE-small: $d_{\text{model}} = 384$, $N = 6$.

### 5.3 Multi-stage обучение GTE

GTE обучается в **три этапа**.

**Этап 1: contrastive pre-training.**

- Данные: 300M+ пар.
- Loss: InfoNCE с in-batch negatives.
- Batch size: 8192.
- Temperature: $\tau = 0.02$.

**Этап 2: обучение с hard negatives.**

- Данные: те же + hard negatives из BM25.
- Loss: InfoNCE + hard negatives.
- Batch size: 1024.

**Этап 3: fine-tuning.**

- Данные: MS MARCO, NQ, etc.
- Loss: InfoNCE + hard negatives.
- Batch size: 256.

### 5.4 Cross-batch negatives в GTE

GTE использует **cross-batch negatives** — negatives из предыдущих батчей. Это увеличивает число negatives без увеличения батча.

**Формально:** пусть $\mathcal{Q}$ — очередь из $K$ эмбеддингов документов. Для query $u_i$:

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\exp(u_i^\top v_i / \tau) + \sum_{v_j \in \mathcal{Q}} \exp(u_i^\top v_j / \tau)}.
$$

**Тонкий момент:** очередь обновляется: старые эмбеддинги удаляются, новые добавляются. Это позволяет использовать тысячи negatives без увеличения батча.

### 5.5 Формулы GTE

**Кодирование:**

$$
u = \text{Normalize}\left( \frac{1}{n} \sum_{i=1}^{n} \text{BERT}(q)_i \right).
$$

**InfoNCE loss с cross-batch negatives:**

$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\exp(u_i^\top v_i / \tau) + \sum_{v_j \in \mathcal{Q}} \exp(u_i^\top v_j / \tau)}.
$$

### 5.6 Результаты GTE

**MTEB:**

| Модель | Средний балл | Retrieval | STS | Classification |
|--------|--------------|-----------|-----|----------------|
| BGE-large | 64.0 | 52.0 | 83.0 | 72.0 |
| GTE-large | 65.0 | 53.0 | 84.0 | 73.0 |
| GTE-base | 63.0 | 51.0 | 82.0 | 71.0 |

**BEIR:**

| Модель | nDCG@10 |
|--------|---------|
| BGE-large | 52.0 |
| GTE-large | 53.0 |

**Наблюдение:** GTE превосходит BGE на MTEB и BEIR.

---

## 6. Сравнение моделей

### 6.1 Сводная таблица

| Модель | Датасет | Batch size | Temperature | Этапы | Hard negatives |
|--------|---------|------------|-------------|-------|----------------|
| E5 | CCPairs (270M) | 32,768 | 0.01 | 2 | BM25 + cross-encoder |
| BGE | 1B+ пар | 8192 | 0.01 | 3 | BM25 + cross-encoder |
| Instructor | Несколько | 1024 | 0.02 | 1 | In-batch |
| GTE | 300M+ пар | 8192 | 0.02 | 3 | BM25 + cross-encoder |

### 6.2 MTEB результаты

| Модель | Средний | Retrieval | STS | Classification | Clustering |
|--------|---------|-----------|-----|----------------|------------|
| SBERT | 52.0 | 40.0 | 75.0 | 60.0 | 45.0 |
| E5-large | 62.0 | 50.0 | 82.0 | 70.0 | 55.0 |
| BGE-large | 64.0 | 52.0 | 83.0 | 72.0 | 57.0 |
| GTE-large | 65.0 | 53.0 | 84.0 | 73.0 | 58.0 |
| Instructor-xl | 64.0 | 52.0 | 83.0 | 74.0 | 58.0 |

### 6.3 BEIR результаты (zero-shot retrieval)

| Модель | nDCG@10 |
|--------|---------|
| BM25 | 40.0 |
| SBERT | 42.0 |
| E5-large | 49.0 |
| BGE-large | 52.0 |
| GTE-large | 53.0 |

---

## 7. Практические рекомендации

### 7.1 Как выбрать модель

**Для английского языка:**

- **GTE-large** — лучшее качество, но медленнее.
- **BGE-large** — хорошее качество, быстрее.
- **E5-large** — хорошее качество, но требует instruction prefix.

**Для русского языка:**

- **ruBERT-based SBERT** — есть русские версии.
- **multilingual-e5-large** — поддерживает русский.
- **bge-m3** — поддерживает 100+ языков.

**Для production:**

- **BGE-small** — быстро, 384 размерность.
- **GTE-small** — быстро, 384 размерность.
- **all-MiniLM-L6-v2** — очень быстро, 384 размерность.

### 7.2 Instruction prefix

**E5:**

```
query: как приготовить пасту
passage: рецепт спагетти
```

**Instructor:**

```
Represent the query for retrieving supporting documents: как приготовить пасту
Represent the document for retrieval: рецепт спагетти
```

**BGE:** не требует префикса.

**Тонкий момент:** если вы используете E5 или Instructor, **обязательно** добавляйте префикс. Без префикса качество падает.

### 7.3 Нормализация

Все современные модели нормализуют эмбеддинги по L2. После нормализации косинусная близость эквивалентна скалярному произведению.

### 7.4 Размерность

| Модель | Размерность |
|--------|-------------|
| E5-base | 768 |
| E5-large | 1024 |
| BGE-base | 768 |
| BGE-large | 1024 |
| BGE-small | 384 |
| GTE-base | 768 |
| GTE-large | 1024 |
| GTE-small | 384 |
| Instructor-base | 768 |
| Instructor-xl | 1024 |

**Тонкий момент:** для больших баз данных (миллионы документов) лучше использовать модели с меньшей размерностью (384), чтобы уменьшить память и ускорить поиск.

### 7.5 Квантизация

Для ускорения inference можно использовать **квантизацию**:

- **float16:** 2x ускорение, минимальная потеря качества.
- **int8:** 4x ускорение, небольшая потеря качества.
- **binary:** 32x ускорение, значительная потеря качества.

**Пример:** BGE-large с int8 квантизацией работает в 4 раза быстрее, но теряет 1–2% качества.

---

## 8. Заключение

Современные модели эмбеддингов (E5, BGE, Instructor, GTE) — это результат эволюции SBERT. Ключевые изменения:

1. **Огромные датасеты:** 270M–1B пар вместо 1M.
2. **Multi-stage обучение:** 2–3 этапа с постепенным усложнением.
3. **Hard negatives:** BM25 + cross-encoder.
4. **Instruction prefix:** для различения ролей.
5. **Большие батчи:** 8192–32768.
6. **Cross-batch negatives:** для увеличения числа negatives.

**Ключевые формулы:**

InfoNCE loss:
$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j=1}^{N} \exp(u_i^\top v_j / \tau)}.
$$

InfoNCE с hard negatives:
$$
\mathcal{L}_i = -\log \frac{\exp(u_i^\top v_i / \tau)}{\exp(u_i^\top v_i / \tau) + \sum_{k} \exp(u_i^\top v_k^- / \tau) + \sum_{j \neq i} \exp(u_i^\top v_j / \tau)}.
$$

Mean pooling:
$$
u = \frac{1}{n} \sum_{i=1}^{n} h_i.
$$

Нормализация:
$$
u \leftarrow \frac{u}{\|u\|_2}.
$$

**Что дальше:** в следующей части мы разберём **MTEB, BEIR и практику** — как оценивать модели, как выбирать, как использовать в RAG и поиске.


# Численный пример современных моделей эмбеддингов: E5, BGE, Instructor

## Полный пошаговый разбор с формулами

---

## 1. Постановка задачи

Мы разберём, как работают современные модели (E5, BGE, Instructor) на конкретном примере. Поскольку все они используют **одну архитектуру** (BERT encoder + mean pooling + contrastive learning), мы построим **унифицированный пример**, который показывает все ключевые шаги.

### 1.1 Данные

Рассмотрим батч из $N = 2$ пар (query, passage):

| Пара | Query | Passage |
|------|-------|---------|
| 1 | «как приготовить пасту» | «рецепт спагетти» |
| 2 | «прогноз погоды» | «завтра будет дождь» |

### 1.2 Параметры

- $d_{\text{model}} = 4$ (размерность эмбеддинга);
- $N = 2$ (размер батча);
- $\tau = 0.1$ (temperature);
- $\eta = 0.01$ (learning rate);
- pooling: mean pooling;
- нормализация: L2.

**Тонкий момент:** в реальных моделях $d_{\text{model}} = 768$ (base) или $1024$ (large), $N = 256$–$8192$, $\tau = 0.01$–$0.02$. Мы используем маленькие значения, чтобы все вычисления можно было проверить вручную.

---

## 2. Instruction Prefix (E5, Instructor)

### 2.1 E5: префиксы `query:` и `passage:`

E5 использует специальные префиксы, чтобы модель знала роль текста:

**Query 1:**

$$
\text{query: как приготовить пасту}
$$

**Passage 1:**

$$
\text{passage: рецепт спагетти}
$$

**Query 2:**

$$
\text{query: прогноз погоды}
$$

**Passage 2:**

$$
\text{passage: завтра будет дождь}
$$

### 2.2 Instructor: инструкции

Instructor использует более длинные инструкции:

**Query 1:**

$$
\text{Represent the query for retrieving supporting documents: как приготовить пасту}
$$

**Passage 1:**

$$
\text{Represent the document for retrieval: рецепт спагетти}
$$

### 2.3 BGE: без префиксов

BGE не использует префиксы:

**Query 1:**

$$
\text{как приготовить пасту}
$$

**Passage 1:**

$$
\text{рецепт спагетти}
$$

**Тонкий момент:** в нашем примере мы будем использовать **E5-подход** с префиксами, потому что он наиболее явно показывает, как инструкция влияет на эмбеддинг.

---

## 3. Токенизация

### 3.1 Токенизация Query 1

BERT использует WordPiece токенизацию. Добавляем `[CLS]` и `[SEP]`:

$$
\text{[CLS]} \quad \text{query} \quad \text{:} \quad \text{как} \quad \text{приготовить} \quad \text{пасту} \quad \text{[SEP]}
$$

**Длина:** $n_{q_1} = 7$.

### 3.2 Токенизация Passage 1

$$
\text{[CLS]} \quad \text{passage} \quad \text{:} \quad \text{рецепт} \quad \text{спагетти} \quad \text{[SEP]}
$$

**Длина:** $n_{p_1} = 6$.

### 3.3 Токенизация Query 2 и Passage 2

Аналогично:

$$
\text{[CLS]} \quad \text{query} \quad \text{:} \quad \text{прогноз} \quad \text{погоды} \quad \text{[SEP]}
$$

**Длина:** $n_{q_2} = 6$.

$$
\text{[CLS]} \quad \text{passage} \quad \text{:} \quad \text{завтра} \quad \text{будет} \quad \text{дождь} \quad \text{[SEP]}
$$

**Длина:** $n_{p_2} = 7$.

---

## 4. Token Embedding

### 4.1 Словарь токенов

Пусть embedding layer выдал для каждого токена вектор размерности 4:

| Токен | Token Embedding |
|-------|-----------------|
| [CLS] | $(0.1, 0.2, 0.3, 0.4)$ |
| [SEP] | $(0.0, 0.1, 0.2, 0.3)$ |
| query | $(0.5, 0.5, 0.5, 0.5)$ |
| passage | $(0.6, 0.6, 0.6, 0.6)$ |
| : | $(0.0, 0.0, 0.0, 0.0)$ |
| как | $(1.0, 0.0, 0.0, 0.0)$ |
| приготовить | $(0.0, 1.0, 0.0, 0.0)$ |
| пасту | $(0.0, 0.0, 1.0, 0.0)$ |
| рецепт | $(0.0, 0.0, 0.0, 1.0)$ |
| спагетти | $(1.0, 1.0, 0.0, 0.0)$ |
| прогноз | $(0.0, 1.0, 1.0, 0.0)$ |
| погоды | $(0.0, 0.0, 1.0, 1.0)$ |
| завтра | $(1.0, 0.0, 0.0, 1.0)$ |
| будет | $(0.5, 0.0, 0.5, 0.0)$ |
| дождь | $(0.0, 0.5, 0.0, 0.5)$ |

### 4.2 Token Embedding для Query 1

$$
E_{\text{token}}^{q_1} = \begin{pmatrix}
0.1 & 0.2 & 0.3 & 0.4 \\
0.5 & 0.5 & 0.5 & 0.5 \\
0.0 & 0.0 & 0.0 & 0.0 \\
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0 \\
0.0 & 0.0 & 1.0 & 0.0 \\
0.0 & 0.1 & 0.2 & 0.3
\end{pmatrix} \in \mathbb{R}^{7 \times 4}.
$$

### 4.3 Token Embedding для Passage 1

$$
E_{\text{token}}^{p_1} = \begin{pmatrix}
0.1 & 0.2 & 0.3 & 0.4 \\
0.6 & 0.6 & 0.6 & 0.6 \\
0.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 0.0 & 0.0 & 1.0 \\
1.0 & 1.0 & 0.0 & 0.0 \\
0.0 & 0.1 & 0.2 & 0.3
\end{pmatrix} \in \mathbb{R}^{6 \times 4}.
$$

---

## 5. Segment Embedding

E5 и BGE используют **только один** сегмент (нет пар предложений, как в BERT для NSP). Поэтому segment embedding одинаков для всех токенов.

Пусть:

$$
E_{\text{segment}} = (0.1, 0.1, 0.1, 0.1).
$$

**Матрица Segment Embedding для Query 1:**

$$
E_{\text{segment}}^{q_1} = \begin{pmatrix}
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1 \\
0.1 & 0.1 & 0.1 & 0.1
\end{pmatrix}.
$$

**Аналогично для Passage 1.**

**Тонкий момент:** в E5 и BGE segment embedding не играет большой роли, потому что нет пар предложений. Но он всё равно используется.

---

## 6. Position Embedding

BERT использует **обучаемые** позиционные эмбеддинги. Пусть они заданы так:

| Позиция | Position Embedding |
|---------|-------------------|
| 1 | $(0.01, 0.02, 0.03, 0.04)$ |
| 2 | $(0.05, 0.06, 0.07, 0.08)$ |
| 3 | $(0.09, 0.10, 0.11, 0.12)$ |
| 4 | $(0.13, 0.14, 0.15, 0.16)$ |
| 5 | $(0.17, 0.18, 0.19, 0.20)$ |
| 6 | $(0.21, 0.22, 0.23, 0.24)$ |
| 7 | $(0.25, 0.26, 0.27, 0.28)$ |

**Матрица Position Embedding для Query 1 (7 токенов):**

$$
E_{\text{position}}^{q_1} = \begin{pmatrix}
0.01 & 0.02 & 0.03 & 0.04 \\
0.05 & 0.06 & 0.07 & 0.08 \\
0.09 & 0.10 & 0.11 & 0.12 \\
0.13 & 0.14 & 0.15 & 0.16 \\
0.17 & 0.18 & 0.19 & 0.20 \\
0.21 & 0.22 & 0.23 & 0.24 \\
0.25 & 0.26 & 0.27 & 0.28
\end{pmatrix}.
$$

**Матрица Position Embedding для Passage 1 (6 токенов):**

$$
E_{\text{position}}^{p_1} = \begin{pmatrix}
0.01 & 0.02 & 0.03 & 0.04 \\
0.05 & 0.06 & 0.07 & 0.08 \\
0.09 & 0.10 & 0.11 & 0.12 \\
0.13 & 0.14 & 0.15 & 0.16 \\
0.17 & 0.18 & 0.19 & 0.20 \\
0.21 & 0.22 & 0.23 & 0.24
\end{pmatrix}.
$$

---

## 7. Сложение трёх эмбеддингов

### 7.1 Для Query 1

$$
\tilde{x}_i^{q_1} = \text{TokenEmbed}(w_i) + \text{SegmentEmbed}(s_i) + \text{PositionEmbed}(i).
$$

**Токен 1 ([CLS]):**

$$
\tilde{x}_1^{q_1} = (0.1, 0.2, 0.3, 0.4) + (0.1, 0.1, 0.1, 0.1) + (0.01, 0.02, 0.03, 0.04).
$$

$$
= (0.21, 0.32, 0.43, 0.54).
$$

**Токен 2 (query):**

$$
\tilde{x}_2^{q_1} = (0.5, 0.5, 0.5, 0.5) + (0.1, 0.1, 0.1, 0.1) + (0.05, 0.06, 0.07, 0.08).
$$

$$
= (0.65, 0.66, 0.67, 0.68).
$$

**Токен 3 (:):**

$$
\tilde{x}_3^{q_1} = (0.0, 0.0, 0.0, 0.0) + (0.1, 0.1, 0.1, 0.1) + (0.09, 0.10, 0.11, 0.12).
$$

$$
= (0.19, 0.20, 0.21, 0.22).
$$

**Токен 4 (как):**

$$
\tilde{x}_4^{q_1} = (1.0, 0.0, 0.0, 0.0) + (0.1, 0.1, 0.1, 0.1) + (0.13, 0.14, 0.15, 0.16).
$$

$$
= (1.23, 0.24, 0.25, 0.26).
$$

**Токен 5 (приготовить):**

$$
\tilde{x}_5^{q_1} = (0.0, 1.0, 0.0, 0.0) + (0.1, 0.1, 0.1, 0.1) + (0.17, 0.18, 0.19, 0.20).
$$

$$
= (0.27, 1.28, 0.29, 0.30).
$$

**Токен 6 (пасту):**

$$
\tilde{x}_6^{q_1} = (0.0, 0.0, 1.0, 0.0) + (0.1, 0.1, 0.1, 0.1) + (0.21, 0.22, 0.23, 0.24).
$$

$$
= (0.31, 0.32, 1.33, 0.34).
$$

**Токен 7 ([SEP]):**

$$
\tilde{x}_7^{q_1} = (0.0, 0.1, 0.2, 0.3) + (0.1, 0.1, 0.1, 0.1) + (0.25, 0.26, 0.27, 0.28).
$$

$$
= (0.35, 0.46, 0.57, 0.68).
$$

**Матрица входа для Query 1:**

$$
\tilde{X}^{q_1} = \begin{pmatrix}
0.21 & 0.32 & 0.43 & 0.54 \\
0.65 & 0.66 & 0.67 & 0.68 \\
0.19 & 0.20 & 0.21 & 0.22 \\
1.23 & 0.24 & 0.25 & 0.26 \\
0.27 & 1.28 & 0.29 & 0.30 \\
0.31 & 0.32 & 1.33 & 0.34 \\
0.35 & 0.46 & 0.57 & 0.68
\end{pmatrix} \in \mathbb{R}^{7 \times 4}.
$$

### 7.2 Для Passage 1

**Токен 1 ([CLS]):**

$$
\tilde{x}_1^{p_1} = (0.21, 0.32, 0.43, 0.54).
$$

**Токен 2 (passage):**

$$
\tilde{x}_2^{p_1} = (0.6, 0.6, 0.6, 0.6) + (0.1, 0.1, 0.1, 0.1) + (0.05, 0.06, 0.07, 0.08).
$$

$$
= (0.75, 0.76, 0.77, 0.78).
$$

**Токен 3 (:):**

$$
\tilde{x}_3^{p_1} = (0.19, 0.20, 0.21, 0.22).
$$

**Токен 4 (рецепт):**

$$
\tilde{x}_4^{p_1} = (0.0, 0.0, 0.0, 1.0) + (0.1, 0.1, 0.1, 0.1) + (0.13, 0.14, 0.15, 0.16).
$$

$$
= (0.23, 0.24, 0.25, 1.26).
$$

**Токен 5 (спагетти):**

$$
\tilde{x}_5^{p_1} = (1.0, 1.0, 0.0, 0.0) + (0.1, 0.1, 0.1, 0.1) + (0.17, 0.18, 0.19, 0.20).
$$

$$
= (1.27, 1.28, 0.29, 0.30).
$$

**Токен 6 ([SEP]):**

$$
\tilde{x}_6^{p_1} = (0.0, 0.1, 0.2, 0.3) + (0.1, 0.1, 0.1, 0.1) + (0.21, 0.22, 0.23, 0.24).
$$

$$
= (0.31, 0.42, 0.53, 0.64).
$$

**Матрица входа для Passage 1:**

$$
\tilde{X}^{p_1} = \begin{pmatrix}
0.21 & 0.32 & 0.43 & 0.54 \\
0.75 & 0.76 & 0.77 & 0.78 \\
0.19 & 0.20 & 0.21 & 0.22 \\
0.23 & 0.24 & 0.25 & 1.26 \\
1.27 & 1.28 & 0.29 & 0.30 \\
0.31 & 0.42 & 0.53 & 0.64
\end{pmatrix} \in \mathbb{R}^{6 \times 4}.
$$

---

## 8. BERT Encoder

### 8.1 Multi-Head Self-Attention

**Параметры (зададим вручную):**

Голова 1:

$$
W_1^Q = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}, \quad
W_1^K = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}, \quad
W_1^V = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}.
$$

Голова 2:

$$
W_2^Q = \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad
W_2^K = \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad
W_2^V = \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}.
$$

**Голова 1 для Query 1: проекции.**

Голова 1 берёт первые две компоненты входа:

$$
Q_1^{q_1} = \tilde{X}^{q_1} W_1^Q = \begin{pmatrix}
0.21 & 0.32 \\
0.65 & 0.66 \\
0.19 & 0.20 \\
1.23 & 0.24 \\
0.27 & 1.28 \\
0.31 & 0.32 \\
0.35 & 0.46
\end{pmatrix} \in \mathbb{R}^{7 \times 2}.
$$

$$
K_1^{q_1} = Q_1^{q_1}, \quad V_1^{q_1} = Q_1^{q_1}.
$$

**Голова 1: attention.**

$$
Q_1^{q_1} (K_1^{q_1})^\top \in \mathbb{R}^{7 \times 7}.
$$

Вычислим несколько элементов:

Элемент (1,1): $0.21^2 + 0.32^2 = 0.0441 + 0.1024 = 0.1465$.

Элемент (1,2): $0.21 \cdot 0.65 + 0.32 \cdot 0.66 = 0.1365 + 0.2112 = 0.3477$.

Элемент (1,4): $0.21 \cdot 1.23 + 0.32 \cdot 0.24 = 0.2583 + 0.0768 = 0.3351$.

Элемент (4,4): $1.23^2 + 0.24^2 = 1.5129 + 0.0576 = 1.5705$.

**Масштабирование на $\sqrt{d_k} = \sqrt{2} \approx 1.4142$:**

$$
\frac{Q_1^{q_1} (K_1^{q_1})^\top}{\sqrt{2}}.
$$

Для элемента (1,1): $0.1465 / 1.4142 = 0.1036$.

Для элемента (4,4): $1.5705 / 1.4142 = 1.1105$.

**Softmax по строкам.**

Для строки 4 (токен «как»):

Значения: $s_{41} / \sqrt{2} = 0.3351 / 1.4142 = 0.2370$, $s_{42} / \sqrt{2} = (1.23 \cdot 0.65 + 0.24 \cdot 0.66) / 1.4142 = (0.7995 + 0.1584) / 1.4142 = 0.9579 / 1.4142 = 0.6774$, $s_{43} / \sqrt{2} = (1.23 \cdot 0.19 + 0.24 \cdot 0.20) / 1.4142 = (0.2337 + 0.0480) / 1.4142 = 0.2817 / 1.4142 = 0.1992$, $s_{44} / \sqrt{2} = 1.1105$.

И так далее.

**Для краткости приведём итоговый результат головы 1 для Query 1:**

$$
\text{head}_1^{q_1} \approx \begin{pmatrix}
0.31 & 0.42 \\
0.62 & 0.65 \\
0.22 & 0.24 \\
0.85 & 0.60 \\
0.55 & 0.95 \\
0.60 & 0.85 \\
0.40 & 0.50
\end{pmatrix}.
$$

**Голова 2 для Query 1: проекции.**

Голова 2 берёт последние две компоненты:

$$
Q_2^{q_1} = \begin{pmatrix}
0.43 & 0.54 \\
0.67 & 0.68 \\
0.21 & 0.22 \\
0.25 & 0.26 \\
0.29 & 0.30 \\
1.33 & 0.34 \\
0.57 & 0.68
\end{pmatrix}.
$$

**Аналогично вычисляем attention и получаем:**

$$
\text{head}_2^{q_1} \approx \begin{pmatrix}
0.48 & 0.52 \\
0.70 & 0.72 \\
0.22 & 0.24 \\
0.30 & 0.32 \\
0.35 & 0.38 \\
0.80 & 0.85 \\
0.60 & 0.65
\end{pmatrix}.
$$

**Конкатенация голов для Query 1:**

$$
\text{Concat}^{q_1} = \begin{pmatrix}
0.31 & 0.42 & 0.48 & 0.52 \\
0.62 & 0.65 & 0.70 & 0.72 \\
0.22 & 0.24 & 0.22 & 0.24 \\
0.85 & 0.60 & 0.30 & 0.32 \\
0.55 & 0.95 & 0.35 & 0.38 \\
0.60 & 0.85 & 0.80 & 0.85 \\
0.40 & 0.50 & 0.60 & 0.65
\end{pmatrix} \in \mathbb{R}^{7 \times 4}.
$$

**Проекция выхода.** $W^O = I_4$. Тогда $\text{MultiHead}^{q_1} = \text{Concat}^{q_1}$.

### 8.2 Residual + Layer Normalization для Query 1

**Residual:**

$$
\tilde{X}^{q_1} + \text{MultiHead}^{q_1} = \begin{pmatrix}
0.21+0.31 & 0.32+0.42 & 0.43+0.48 & 0.54+0.52 \\
0.65+0.62 & 0.66+0.65 & 0.67+0.70 & 0.68+0.72 \\
0.19+0.22 & 0.20+0.24 & 0.21+0.22 & 0.22+0.24 \\
1.23+0.85 & 0.24+0.60 & 0.25+0.30 & 0.26+0.32 \\
0.27+0.55 & 1.28+0.95 & 0.29+0.35 & 0.30+0.38 \\
0.31+0.60 & 0.32+0.85 & 1.33+0.80 & 0.34+0.85 \\
0.35+0.40 & 0.46+0.50 & 0.57+0.60 & 0.68+0.65
\end{pmatrix}.
$$

$$
= \begin{pmatrix}
0.52 & 0.74 & 0.91 & 1.06 \\
1.27 & 1.31 & 1.37 & 1.40 \\
0.41 & 0.44 & 0.43 & 0.46 \\
2.08 & 0.84 & 0.55 & 0.58 \\
0.82 & 2.23 & 0.64 & 0.68 \\
0.91 & 1.17 & 2.13 & 1.19 \\
0.75 & 0.96 & 1.17 & 1.33
\end{pmatrix}.
$$

**Layer Normalization (строка 1):**

Среднее:

$$
\mu_1 = \frac{0.52 + 0.74 + 0.91 + 1.06}{4} = \frac{3.23}{4} = 0.8075.
$$

Дисперсия:

$$
\sigma_1^2 = \frac{(0.52-0.8075)^2 + (0.74-0.8075)^2 + (0.91-0.8075)^2 + (1.06-0.8075)^2}{4}.
$$

$$
= \frac{0.0827 + 0.0046 + 0.0105 + 0.0638}{4} = \frac{0.1616}{4} = 0.0404.
$$

$$
\sqrt{0.0404 + 10^{-5}} \approx 0.2010.
$$

$$
\hat{x}_1^{q_1} = \left( \frac{-0.2875}{0.2010}, \frac{-0.0675}{0.2010}, \frac{0.1025}{0.2010}, \frac{0.2525}{0.2010} \right) = (-1.4303, -0.3358, 0.5100, 1.2562).
$$

**Для остальных строк аналогично.** Приведём итог:

$$
Z^{q_1} = \begin{pmatrix}
-1.4303 & -0.3358 & 0.5100 & 1.2562 \\
-0.6419 & 0.0990 & 0.5640 & 0.9789 \\
-1.5300 & 0.4200 & -0.5400 & 1.6500 \\
1.4900 & -0.6100 & -1.0900 & 0.2100 \\
-0.5600 & 1.6300 & -0.4300 & -0.6400 \\
-0.8600 & 0.3100 & 1.3800 & -0.8300 \\
-0.8900 & 0.0800 & 0.7100 & 1.1000
\end{pmatrix}.
$$

### 8.3 Feed-Forward Network для Query 1

**Параметры:** $d_{ff} = 8$. Зададим $W_1 \in \mathbb{R}^{4 \times 8}$ и $W_2 \in \mathbb{R}^{8 \times 4}$ с малыми значениями.

**Для простоты используем:** $W_1$ — матрица, где каждая строка — $(0.1, 0.1, \ldots, 0.1)$ (8 раз). Тогда:

$$
z_1 = Z^{q_1} W_1.
$$

**Строка 1:**

$$
z_{1,1} = 0.1 \cdot (-1.4303 - 0.3358 + 0.5100 + 1.2562) = 0.1 \cdot 0.0001 = 0.00001.
$$

Все 8 компонент $z_1$ близки к 0.

**ReLU:** $\text{ReLU}(z_1) \approx 0$.

**Выход FFN:** $F^{q_1} \approx 0$.

**Тонкий момент:** в реальных моделях FFN даёт значимые значения. У нас веса заданы вручную, поэтому ReLU «обнуляет» компоненты.

### 8.4 Residual + Layer Normalization для Query 1

$$
Z^{q_1} + F^{q_1} \approx Z^{q_1}.
$$

**Layer Normalization:** аналогично предыдущему. Для краткости приведём итог:

$$
\text{Output}^{q_1} \approx Z^{q_1}.
$$

**Аналогично для Passage 1, Query 2, Passage 2.** Для краткости приведём итоговые выходы BERT:

**Passage 1:**

$$
\text{Output}^{p_1} \approx \begin{pmatrix}
-1.4200 & -0.3400 & 0.5100 & 1.2500 \\
-0.5100 & 0.1200 & 0.5800 & 0.9700 \\
-1.5200 & 0.4300 & -0.5500 & 1.6400 \\
-0.6500 & -0.7500 & -0.8500 & 1.2500 \\
1.4500 & 1.4800 & -0.6200 & -0.3100 \\
-0.8400 & 0.3200 & 1.3700 & -0.8500
\end{pmatrix}.
$$

**Query 2:**

$$
\text{Output}^{q_2} \approx \begin{pmatrix}
-1.4100 & -0.3300 & 0.5200 & 1.2400 \\
-0.6300 & 0.1000 & 0.5700 & 0.9800 \\
-1.5300 & 0.4400 & -0.5600 & 1.6500 \\
-0.5500 & 1.6200 & -0.4400 & -0.6300 \\
-0.8700 & 0.3200 & 1.3700 & -0.8400 \\
-0.8900 & 0.0900 & 0.7200 & 1.1000
\end{pmatrix}.
$$

**Passage 2:**

$$
\text{Output}^{p_2} \approx \begin{pmatrix}
-1.4000 & -0.3200 & 0.5300 & 1.2300 \\
-0.5200 & 0.1300 & 0.5900 & 0.9600 \\
-1.5100 & 0.4500 & -0.5700 & 1.6300 \\
-0.6400 & -0.7400 & -0.8600 & 1.2400 \\
-0.8500 & 0.3100 & 1.3600 & -0.8600 \\
-0.8800 & 0.0700 & 0.7300 & 1.0800 \\
-0.9000 & 0.0600 & 0.7400 & 1.0700
\end{pmatrix}.
$$

---

## 9. Mean Pooling

### 9.1 Mean Pooling для Query 1

$$
u_1 = \frac{1}{n_{q_1}} \sum_{i=1}^{n_{q_1}} \text{Output}_i^{q_1}.
$$

$n_{q_1} = 7$:

$$
u_1 = \frac{1}{7} \left[ (-1.4303, -0.3358, 0.5100, 1.2562) + (-0.6419, 0.0990, 0.5640, 0.9789) + (-1.5300, 0.4200, -0.5400, 1.6500) + (1.4900, -0.6100, -1.0900, 0.2100) + (-0.5600, 1.6300, -0.4300, -0.6400) + (-0.8600, 0.3100, 1.3800, -0.8300) + (-0.8900, 0.0800, 0.7100, 1.1000) \right].
$$

Сумма:

Первая компонента:

$$
-1.4303 - 0.6419 - 1.5300 + 1.4900 - 0.5600 - 0.8600 - 0.8900 = -4.4222.
$$

Вторая компонента:

$$
-0.3358 + 0.0990 + 0.4200 - 0.6100 + 1.6300 + 0.3100 + 0.0800 = 1.5932.
$$

Третья компонента:

$$
0.5100 + 0.5640 - 0.5400 - 1.0900 - 0.4300 + 1.3800 + 0.7100 = 1.1040.
$$

Четвёртая компонента:

$$
1.2562 + 0.9789 + 1.6500 + 0.2100 - 0.6400 - 0.8300 + 1.1000 = 3.7251.
$$

$$
u_1 = \frac{1}{7} (-4.4222, 1.5932, 1.1040, 3.7251) = (-0.6317, 0.2276, 0.1577, 0.5322).
$$

### 9.2 Mean Pooling для Passage 1

$n_{p_1} = 6$:

$$
v_1 = \frac{1}{6} \left[ (-1.4200, -0.3400, 0.5100, 1.2500) + (-0.5100, 0.1200, 0.5800, 0.9700) + (-1.5200, 0.4300, -0.5500, 1.6400) + (-0.6500, -0.7500, -0.8500, 1.2500) + (1.4500, 1.4800, -0.6200, -0.3100) + (-0.8400, 0.3200, 1.3700, -0.8500) \right].
$$

Сумма:

Первая компонента:

$$
-1.4200 - 0.5100 - 1.5200 - 0.6500 + 1.4500 - 0.8400 = -3.4900.
$$

Вторая компонента:

$$
-0.3400 + 0.1200 + 0.4300 - 0.7500 + 1.4800 + 0.3200 = 1.2600.
$$

Третья компонента:

$$
0.5100 + 0.5800 - 0.5500 - 0.8500 - 0.6200 + 1.3700 = 0.4400.
$$

Четвёртая компонента:

$$
1.2500 + 0.9700 + 1.6400 + 1.2500 - 0.3100 - 0.8500 = 3.9500.
$$

$$
v_1 = \frac{1}{6} (-3.4900, 1.2600, 0.4400, 3.9500) = (-0.5817, 0.2100, 0.0733, 0.6583).
$$

### 9.3 Mean Pooling для Query 2

$n_{q_2} = 6$:

$$
u_2 = \frac{1}{6} \left[ (-1.4100, -0.3300, 0.5200, 1.2400) + (-0.6300, 0.1000, 0.5700, 0.9800) + (-1.5300, 0.4400, -0.5600, 1.6500) + (-0.5500, 1.6200, -0.4400, -0.6300) + (-0.8700, 0.3200, 1.3700, -0.8400) + (-0.8900, 0.0900, 0.7200, 1.1000) \right].
$$

Сумма:

Первая компонента:

$$
-1.4100 - 0.6300 - 1.5300 - 0.5500 - 0.8700 - 0.8900 = -5.8800.
$$

Вторая компонента:

$$
-0.3300 + 0.1000 + 0.4400 + 1.6200 + 0.3200 + 0.0900 = 2.2400.
$$

Третья компонента:

$$
0.5200 + 0.5700 - 0.5600 - 0.4400 + 1.3700 + 0.7200 = 2.1800.
$$

Четвёртая компонента:

$$
1.2400 + 0.9800 + 1.6500 - 0.6300 - 0.8400 + 1.1000 = 3.5000.
$$

$$
u_2 = \frac{1}{6} (-5.8800, 2.2400, 2.1800, 3.5000) = (-0.9800, 0.3733, 0.3633, 0.5833).
$$

### 9.4 Mean Pooling для Passage 2

$n_{p_2} = 7$:

$$
v_2 = \frac{1}{7} \left[ (-1.4000, -0.3200, 0.5300, 1.2300) + (-0.5200, 0.1300, 0.5900, 0.9600) + (-1.5100, 0.4500, -0.5700, 1.6300) + (-0.6400, -0.7400, -0.8600, 1.2400) + (-0.8500, 0.3100, 1.3600, -0.8600) + (-0.8800, 0.0700, 0.7300, 1.0800) + (-0.9000, 0.0600, 0.7400, 1.0700) \right].
$$

Сумма:

Первая компонента:

$$
-1.4000 - 0.5200 - 1.5100 - 0.6400 - 0.8500 - 0.8800 - 0.9000 = -6.7000.
$$

Вторая компонента:

$$
-0.3200 + 0.1300 + 0.4500 - 0.7400 + 0.3100 + 0.0700 + 0.0600 = -0.0400.
$$

Третья компонента:

$$
0.5300 + 0.5900 - 0.5700 - 0.8600 + 1.3600 + 0.7300 + 0.7400 = 2.5200.
$$

Четвёртая компонента:

$$
1.2300 + 0.9600 + 1.6300 + 1.2400 - 0.8600 + 1.0800 + 1.0700 = 6.3500.
$$

$$
v_2 = \frac{1}{7} (-6.7000, -0.0400, 2.5200, 6.3500) = (-0.9571, -0.0057, 0.3600, 0.9071).
$$

---

## 10. Нормализация эмбеддингов

### 10.1 Нормализация $u_1$

$$
\|u_1\|_2 = \sqrt{(-0.6317)^2 + (0.2276)^2 + (0.1577)^2 + (0.5322)^2}.
$$

$$
= \sqrt{0.3990 + 0.0518 + 0.0249 + 0.2832} = \sqrt{0.7589} \approx 0.8711.
$$

$$
u_1^{\text{norm}} = \frac{(-0.6317, 0.2276, 0.1577, 0.5322)}{0.8711} = (-0.7252, 0.2613, 0.1810, 0.6110).
$$

### 10.2 Нормализация $v_1$

$$
\|v_1\|_2 = \sqrt{(-0.5817)^2 + (0.2100)^2 + (0.0733)^2 + (0.6583)^2}.
$$

$$
= \sqrt{0.3384 + 0.0441 + 0.0054 + 0.4334} = \sqrt{0.8213} \approx 0.9063.
$$

$$
v_1^{\text{norm}} = \frac{(-0.5817, 0.2100, 0.0733, 0.6583)}{0.9063} = (-0.6418, 0.2317, 0.0809, 0.7264).
$$

### 10.3 Нормализация $u_2$

$$
\|u_2\|_2 = \sqrt{(-0.9800)^2 + (0.3733)^2 + (0.3633)^2 + (0.5833)^2}.
$$

$$
= \sqrt{0.9604 + 0.1394 + 0.1320 + 0.3403} = \sqrt{1.5721} \approx 1.2538.
$$

$$
u_2^{\text{norm}} = \frac{(-0.9800, 0.3733, 0.3633, 0.5833)}{1.2538} = (-0.7816, 0.2977, 0.2898, 0.4652).
$$

### 10.4 Нормализация $v_2$

$$
\|v_2\|_2 = \sqrt{(-0.9571)^2 + (-0.0057)^2 + (0.3600)^2 + (0.9071)^2}.
$$

$$
= \sqrt{0.9160 + 0.0000 + 0.1296 + 0.8228} = \sqrt{1.8684} \approx 1.3669.
$$

$$
v_2^{\text{norm}} = \frac{(-0.9571, -0.0057, 0.3600, 0.9071)}{1.3669} = (-0.7002, -0.0042, 0.2634, 0.6636).
$$

---

## 11. Similarity Matrix

### 11.1 Вычисление $s_{ij} = u_i^\top v_j$

**Строка 1 ($u_1$ = «как приготовить пасту»):**

$$
s_{11} = u_1^\top v_1 = (-0.7252)(-0.6418) + (0.2613)(0.2317) + (0.1810)(0.0809) + (0.6110)(0.7264).
$$

$$
= 0.4654 + 0.0605 + 0.0146 + 0.4438 = 0.9843.
$$

$$
s_{12} = u_1^\top v_2 = (-0.7252)(-0.7002) + (0.2613)(-0.0042) + (0.1810)(0.2634) + (0.6110)(0.6636).
$$

$$
= 0.5078 - 0.0011 + 0.0477 + 0.4055 = 0.9599.
$$

**Строка 2 ($u_2$ = «прогноз погоды»):**

$$
s_{21} = u_2^\top v_1 = (-0.7816)(-0.6418) + (0.2977)(0.2317) + (0.2898)(0.0809) + (0.4652)(0.7264).
$$

$$
= 0.5017 + 0.0690 + 0.0234 + 0.3379 = 0.9320.
$$

$$
s_{22} = u_2^\top v_2 = (-0.7816)(-0.7002) + (0.2977)(-0.0042) + (0.2898)(0.2634) + (0.4652)(0.6636).
$$

$$
= 0.5473 - 0.0013 + 0.0763 + 0.3087 = 0.9310.
$$

### 11.2 Матрица similarity

$$
S = \begin{pmatrix}
0.9843 & 0.9599 \\
0.9320 & 0.9310
\end{pmatrix}.
$$

**Наблюдение:** все similarity высокие (0.93–0.98). Это потому что в нашем примере эмбеддинги не очень хорошо разделены (веса заданы вручную). В реальных моделях positive имеет similarity 0.7–0.9, а negatives — 0.1–0.3.

**Тонкий момент:** hard negative здесь — это $s_{12} = 0.9599$ (запрос «как приготовить пасту» и документ «завтра будет дождь»). Это ложная связь, потому что эмбеддинги не обучены.

---

## 12. InfoNCE Loss

### 12.1 Loss для $i = 1$

$$
\mathcal{L}_1 = -\log \frac{\exp(s_{11} / \tau)}{\sum_{j=1}^{2} \exp(s_{1j} / \tau)}.
$$

$\tau = 0.1$:

$$
s_{11} / \tau = 9.843, \quad s_{12} / \tau = 9.599.
$$

$$
\exp(9.843) = 18830.5, \quad \exp(9.599) = 14761.0.
$$

$$
Z_1 = 18830.5 + 14761.0 = 33591.5.
$$

$$
P_{11} = \frac{18830.5}{33591.5} = 0.5606.
$$

$$
\mathcal{L}_1 = -\log(0.5606) = 0.5790.
$$

### 12.2 Loss для $i = 2$

$$
s_{21} / \tau = 9.320, \quad s_{22} / \tau = 9.310.
$$

$$
\exp(9.320) = 11155.0, \quad \exp(9.310) = 11044.0.
$$

$$
Z_2 = 11155.0 + 11044.0 = 22199.0.
$$

$$
P_{22} = \frac{11044.0}{22199.0} = 0.4975.
$$

$$
\mathcal{L}_2 = -\log(0.4975) = 0.6983.
$$

### 12.3 Общий loss

$$
\mathcal{L} = \frac{1}{2} (0.5790 + 0.6983) = 0.6387.
$$

**Наблюдение:** loss $\approx 0.64$. Это высокое значение, потому что эмбеддинги не разделены (positive и negatives имеют похожие similarity). В реальных моделях после обучения loss был бы близок к 0.

---

## 13. Градиенты и обновление весов

### 13.1 Градиент по $u_1$

Формула:

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{1j} v_j - v_1 \right).
$$

**Шаг 1: вычисление $P_{1j}$.**

$$
P_{11} = 0.5606, \quad P_{12} = \frac{14761.0}{33591.5} = 0.4394.
$$

**Шаг 2: взвешенная сумма.**

$$
\sum_{j=1}^{2} P_{1j} v_j = 0.5606 \cdot v_1 + 0.4394 \cdot v_2.
$$

Подставляем $v_1 = (-0.6418, 0.2317, 0.0809, 0.7264)$, $v_2 = (-0.7002, -0.0042, 0.2634, 0.6636)$:

Первая компонента:

$$
0.5606 \cdot (-0.6418) + 0.4394 \cdot (-0.7002) = -0.3598 - 0.3077 = -0.6675.
$$

Вторая компонента:

$$
0.5606 \cdot 0.2317 + 0.4394 \cdot (-0.0042) = 0.1299 - 0.0018 = 0.1281.
$$

Третья компонента:

$$
0.5606 \cdot 0.0809 + 0.4394 \cdot 0.2634 = 0.0453 + 0.1157 = 0.1610.
$$

Четвёртая компонента:

$$
0.5606 \cdot 0.7264 + 0.4394 \cdot 0.6636 = 0.4072 + 0.2916 = 0.6988.
$$

$$
\sum P_{1j} v_j = (-0.6675, 0.1281, 0.1610, 0.6988).
$$

**Шаг 3: вычитание $v_1$.**

$$
\sum P_{1j} v_j - v_1 = (-0.6675 + 0.6418, 0.1281 - 0.2317, 0.1610 - 0.0809, 0.6988 - 0.7264).
$$

$$
= (-0.0257, -0.1036, 0.0801, -0.0276).
$$

**Шаг 4: деление на $\tau$.**

$$
\frac{\partial \mathcal{L}_1}{\partial u_1} = \frac{1}{0.1} (-0.0257, -0.1036, 0.0801, -0.0276) = (-0.257, -1.036, 0.801, -0.276).
$$

### 13.2 Градиент по $v_1$

Формула:

$$
\frac{\partial \mathcal{L}}{\partial v_1} = \frac{1}{\tau} \left( \sum_{i \neq 1} P_{i1} u_i - (1 - P_{11}) u_1 \right).
$$

**Шаг 1: вычисление $P_{21}$.**

$$
P_{21} = \frac{\exp(s_{21} / \tau)}{Z_2} = \frac{11155.0}{22199.0} = 0.5025.
$$

**Шаг 2: взвешенная сумма.**

$$
\sum_{i \neq 1} P_{i1} u_i = 0.5025 \cdot u_2 = 0.5025 \cdot (-0.7816, 0.2977, 0.2898, 0.4652) = (-0.3927, 0.1496, 0.1456, 0.2338).
$$

**Шаг 3: $(1 - P_{11}) u_1$.**

$$
1 - P_{11} = 1 - 0.5606 = 0.4394.
$$

$$
(1 - P_{11}) u_1 = 0.4394 \cdot (-0.7252, 0.2613, 0.1810, 0.6110) = (-0.3187, 0.1148, 0.0795, 0.2685).
$$

**Шаг 4: вычитание.**

$$
\sum P_{i1} u_i - (1 - P_{11}) u_1 = (-0.3927 + 0.3187, 0.1496 - 0.1148, 0.1456 - 0.0795, 0.2338 - 0.2685).
$$

$$
= (-0.0740, 0.0348, 0.0661, -0.0347).
$$

**Шаг 5: деление на $\tau$.**

$$
\frac{\partial \mathcal{L}}{\partial v_1} = \frac{1}{0.1} (-0.0740, 0.0348, 0.0661, -0.0347) = (-0.740, 0.348, 0.661, -0.347).
$$

### 13.3 Обновление $u_1$ и $v_1$

$$
u_1 \leftarrow u_1 - \eta \frac{\partial \mathcal{L}_1}{\partial u_1} = (-0.7252, 0.2613, 0.1810, 0.6110) - 0.01 \cdot (-0.257, -1.036, 0.801, -0.276).
$$

$$
= (-0.7252 + 0.0026, 0.2613 + 0.0104, 0.1810 - 0.0080, 0.6110 + 0.0028) = (-0.7226, 0.2717, 0.1730, 0.6138).
$$

$$
v_1 \leftarrow v_1 - \eta \frac{\partial \mathcal{L}}{\partial v_1} = (-0.6418, 0.2317, 0.0809, 0.7264) - 0.01 \cdot (-0.740, 0.348, 0.661, -0.347).
$$

$$
= (-0.6418 + 0.0074, 0.2317 - 0.0035, 0.0809 - 0.0066, 0.7264 + 0.0035) = (-0.6344, 0.2282, 0.0743, 0.7299).
$$

**Наблюдение:** $u_1$ сдвинулся в направлении $v_1$ (первая компонента стала менее отрицательной, вторая — более положительной), и немного оттолкнулся от $v_2$ (третья компонента уменьшилась).

---

## 14. Сводка всех шагов

| Шаг | Компонент | Что происходит |
|-----|-----------|----------------|
| 1 | Instruction prefix | Добавление `query:` и `passage:` |
| 2 | Токенизация | Разбиение на токены, `[CLS]`, `[SEP]` |
| 3 | Token Embedding | Токены → векторы |
| 4 | Segment Embedding | Добавление информации о сегменте |
| 5 | Position Embedding | Добавление информации о позиции |
| 6 | Сложение | Три эмбеддинга складываются |
| 7 | Multi-Head Self-Attention | Каждое слово видит все остальные |
| 8 | Residual + LayerNorm | Стабилизация |
| 9 | Feed-Forward | Нелинейное преобразование |
| 10 | Residual + LayerNorm | Стабилизация |
| 11 | Mean Pooling | Усреднение выходов BERT |
| 12 | L2-нормализация | Нормализация эмбеддинга |
| 13 | Similarity | $s_{ij} = u_i^\top v_j$ |
| 14 | InfoNCE loss | $-\log P_{ii}$ |
| 15 | Градиенты | Через pooling и BERT |
| 16 | Обновление | $u \leftarrow u - \eta \nabla$ |

---

## 15. Что было упрощено

| Компонент | Реальная модель | Наш пример |
|-----------|-----------------|------------|
| $d_{\text{model}}$ | 768 (base) / 1024 (large) | 4 |
| $N$ (батч) | 256–8192 | 2 |
| $\tau$ | 0.01–0.02 | 0.1 |
| Энкодер | BERT-base/large | Упрощённый |
| Данные | 270M–1B пар | 2 пары |
| Hard negatives | BM25 + cross-encoder | Нет |
| Instruction prefix | `query:` / `passage:` | Да |
| Mean pooling | Да | Да |
| Нормализация | L2 | L2 |

**Тем не менее** пример показывает **все ключевые шаги** современных моделей:

1. Instruction prefix (E5, Instructor).
2. Токенизация с `[CLS]` и `[SEP]`.
3. Три вида эмбеддингов.
4. BERT encoder.
5. Mean pooling.
6. L2-нормализация.
7. Similarity matrix.
8. InfoNCE loss.
9. Градиенты.
10. Обновление весов.

---

## 16. Заключение

В этом численном примере мы шаг за шагом вычислили современный эмбеддинг для двух пар (query, passage). Основные выводы:

1. **Instruction prefix** (E5, Instructor) помогает модели различать роли query и passage.

2. **Mean pooling** агрегирует выходы BERT в один вектор фиксированной размерности.

3. **L2-нормализация** делает косинусную близость эквивалентной скалярному произведению.

4. **InfoNCE loss** заставляет модель различать positive от negatives.

5. **Градиенты** текут через pooling и BERT. В siamese-сети BERT **общий** для query и passage.

6. **Обновление весов** происходит для всех параметров BERT.

**Ключевые формулы:**

Instruction prefix:
$$
\text{input} = \text{query: } q \quad \text{или} \quad \text{passage: } d.
$$

Mean pooling:
$$
u = \frac{1}{n} \sum_{i=1}^{n} h_i.
$$

L2-нормализация:
$$
u \leftarrow \frac{u}{\|u\|_2}.
$$

Similarity:
$$
s_{ij} = u_i^\top v_j.
$$

InfoNCE loss:
$$
\mathcal{L}_i = -\log \frac{\exp(s_{ii} / \tau)}{\sum_{j=1}^{N} \exp(s_{ij} / \tau)}.
$$

Градиент по $u_i$:
$$
\frac{\partial \mathcal{L}_i}{\partial u_i} = \frac{1}{\tau} \left( \sum_{j=1}^{N} P_{ij} v_j - v_i \right).
$$

Этот пример, несмотря на все упрощения, даёт полное понимание того, как работают современные модели эмбеддингов (E5, BGE, Instructor). В реальных моделях $d_{\text{model}} = 768$, $N = 8192$, но математика остаётся той же.


# Часть 3. Оценка эмбеддингов: MTEB, BEIR и практика

## Полная теория с формулами и объяснениями

---

## 1. Введение: зачем нужны бенчмарки

### 1.1 Проблема оценки эмбеддингов

Мы разобрали современные модели эмбеддингов (E5, BGE, Instructor, GTE), но как понять, какая из них **лучше**? Как сравнить модели между собой? Как выбрать модель для конкретной задачи?

**Наивный подход:** обучить модель, запустить на своей задаче, посмотреть метрики. Но это **не масштабируется**: для каждой задачи нужно своё обучение, свои данные, свои метрики. И результат будет зависеть от конкретных данных, а не от «общего качества» модели.

**Решение:** использовать **стандартные бенчмарки** — наборы задач, на которых модели сравниваются в одинаковых условиях. Бенчмарки позволяют:

1. **Сравнивать модели** между собой.
2. **Отслеживать прогресс** в области.
3. **Выбирать модель** для конкретной задачи.
4. **Выявлять слабые места** моделей.

**Два главных бенчмарка для эмбеддингов:**

- **MTEB (Massive Text Embedding Benchmark):** широкий бенчмарк с 56+ датасетами, 7+ типами задач, 112+ языками.
- **BEIR (Benchmarking IR):** узкий бенчмарк, сфокусированный на **zero-shot retrieval** — способности модели искать документы в **новом** домене без обучения.

### 1.2 Исторический контекст

До 2021 года не было стандартного бенчмарка для эмбеддингов. Каждая статья использовала свои данные, свои метрики, свои baseline'ы. Это делало сравнение моделей **невозможным**.

**2021: BEIR.** Nandan Thakur и коллеги предложили BEIR — гетерогенный бенчмарк для zero-shot retrieval. BEIR включает 18 датасетов из разных доменов (медицина, наука, новости, вопросы). Ключевая идея: модель обучается на **одном** датасете (например, MS MARCO), а тестируется на **других** (zero-shot).

**2022: MTEB.** Muennighoff и коллеги предложили MTEB — массовый бенчмарк для эмбеддингов. MTEB включает 56 датасетов, 7 типов задач (classification, clustering, pair classification, reranking, retrieval, STS, summarization), 112 языков. MTEB стал **де-факто** стандартом для оценки эмбеддингов.

**2023–2026: эволюция.** MTEB и BEIR постоянно обновляются: добавляются новые датасеты, языки, задачи. Сегодня MTEB — это не один бенчмарк, а **семейство** бенчмарков: MTEB(Eng, v2), MTEB(Multilingual, v2), MTEB(Code).

---

## 2. MTEB: Massive Text Embedding Benchmark

### 2.1 Общая структура

**MTEB** — это набор задач, каждая из которых проверяет определённый аспект качества эмбеддингов.

**Типы задач в MTEB:**

| Тип задачи | Что проверяет | Метрика |
|------------|---------------|---------|
| **Bitext Mining** | Поиск параллельных предложений в двух языках | F1 |
| **Classification** | Классификация текстов по меткам | Accuracy, F1 |
| **Clustering** | Группировка текстов в кластеры | V-measure |
| **Pair Classification** | Определение, связаны ли два текста | Average Precision |
| **Reranking** | Переранжирование результатов по релевантности | MAP, MRR |
| **Retrieval** | Поиск релевантных документов по запросу | nDCG@10 |
| **STS** | Семантическая близость пар предложений | Spearman correlation |
| **Summarization** | Оценка качества суммаризации | Spearman correlation |

**Тонкий момент:** каждая задача использует **свою** метрику. Это означает, что MTEB — это не один балл, а **набор** баллов. Лидерборд агрегирует их в средний балл, но это не всегда осмысленно: модель может быть хороша в retrieval, но плоха в clustering.

### 2.2 Classification

**Задача:** даны тексты с метками классов. Нужно обучить классификатор на эмбеддингах и оценить его качество.

**Процедура:**

1. Вычислить эмбеддинги для train-набора.
2. Обучить **logistic regression** на эмбеддингах (train).
3. Вычислить эмбеддинги для test-набора.
4. Предсказать метки на test с помощью обученного классификатора.
5. Вычислить метрики.

**Метрики:**

- **Accuracy:** доля правильных предсказаний.

$$
\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}.
$$

- **F1:** гармоническое среднее precision и recall.

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}.
$$

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}.
$$

**Разберём формулу F1 по частям:**

- **Precision:** доля правильных среди всех предсказанных положительных. Если precision = 0.8, то 80% предсказанных «положительных» действительно положительные.
- **Recall:** доля найденных среди всех истинных положительных. Если recall = 0.7, то 70% истинных «положительных» найдены.
- **F1:** баланс между precision и recall. F1 = 1, если precision = recall = 1. F1 = 0, если precision = 0 или recall = 0.

**Тонкий момент:** в MTEB используется **logistic regression** как простой классификатор. Это означает, что MTEB проверяет, насколько эмбеддинги **линейно разделимы** по классам. Если модель даёт хорошие эмбеддинги, logistic regression покажет высокую accuracy.

### 2.3 Clustering

**Задача:** сгруппировать тексты в кластеры так, чтобы похожие тексты были в одном кластере.

**Процедура:**

1. Вычислить эмбеддинги для всех текстов.
2. Запустить **mini-batch k-means** с $k = $ число истинных меток.
3. Вычислить V-measure.

**V-measure:**

$$
V = \frac{2 \cdot H \cdot C}{H + C},
$$

где:

- $H$ — homogeneity: насколько каждый кластер содержит только один класс;
- $C$ — completeness: насколько каждый класс содержится только в одном кластере.

**Homogeneity:**

$$
H = 1 - \frac{H(C \mid K)}{H(C)},
$$

где $H(C \mid K)$ — условная энтропия классов при условии кластеров, $H(C)$ — энтропия классов.

**Completeness:**

$$
C = 1 - \frac{H(K \mid C)}{H(K)},
$$

где $H(K \mid C)$ — условная энтропия кластеров при условии классов, $H(K)$ — энтропия кластеров.

**Разберём интуицию:**

- **Homogeneity = 1:** каждый кластер содержит тексты только одного класса.
- **Completeness = 1:** все тексты одного класса находятся в одном кластере.
- **V-measure = 1:** идеальная кластеризация.

**Тонкий момент:** V-measure не зависит от **названий** кластеров. Если k-means переименует кластеры, V-measure не изменится. Это важно, потому что k-means не знает истинных меток.

### 2.4 Pair Classification

**Задача:** даны пары текстов, нужно определить, связаны ли они (бинарная метка).

**Процедура:**

1. Вычислить эмбеддинги для обоих текстов в паре.
2. Вычислить косинусную близость.
3. Обучить порог на train, применить на test.
4. Вычислить Average Precision.

**Average Precision (AP):**

$$
\text{AP} = \sum_{k=1}^{n} P(k) \cdot \Delta r(k),
$$

где $P(k)$ — precision на первых $k$ элементах, $\Delta r(k)$ — изменение recall.

**Разберём формулу:** AP — это площадь под кривой precision-recall. Чем выше AP, тем лучше модель ранжирует пары.

**Тонкий момент:** в MTEB используется косинусная близость как «оценка» пары. Если близость высокая, пара считается связанной. AP показывает, насколько хорошо близость **ранжирует** пары: связанные пары должны иметь высокую близость, несвязанные — низкую.

### 2.5 Reranking

**Задача:** дан запрос и список документов, нужно **переранжировать** документы по релевантности.

**Процедура:**

1. Вычислить эмбеддинги запроса и документов.
2. Вычислить косинусную близость.
3. Отсортировать документы по близости.
4. Вычислить MAP и MRR.

**Mean Average Precision (MAP):**

$$
\text{MAP} = \frac{1}{Q} \sum_{q=1}^{Q} \text{AP}(q),
$$

где $\text{AP}(q)$ — Average Precision для запроса $q$.

**Mean Reciprocal Rank (MRR):**

$$
\text{MRR} = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{\text{rank}_q},
$$

где $\text{rank}_q$ — позиция первого релевантного документа для запроса $q$.

**Разберём формулу MRR по частям:**

- Если первый релевантный документ на позиции 1: вклад = $1/1 = 1$.
- Если на позиции 2: вклад = $1/2 = 0.5$.
- Если на позиции 10: вклад = $1/10 = 0.1$.

MRR фокусируется на **первом** релевантном документе. Чем выше он в списке, тем больше MRR.

**Тонкий момент:** reranking отличается от retrieval тем, что список документов уже дан. Модель не ищет документы, а только **сортирует** их.

### 2.6 Retrieval

**Задача:** дан запрос и корпус документов, нужно найти релевантные документы.

**Процедура:**

1. Вычислить эмбеддинги запроса и всех документов.
2. Найти top-$k$ документов по косинусной близости.
3. Вычислить nDCG@10.

**Формула nDCG@10:**

$$
\text{nDCG@10} = \frac{\text{DCG@10}}{\text{IDCG@10}},
$$

где:

$$
\text{DCG@10} = \sum_{i=1}^{10} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)},
$$

$$
\text{IDCG@10} = \sum_{i=1}^{10} \frac{2^{\text{rel}_i^*} - 1}{\log_2(i + 1)},
$$

где $\text{rel}_i$ — релевантность документа на позиции $i$, $\text{rel}_i^*$ — релевантность в **идеальном** ранжировании.

**Разберём формулу DCG по частям:**

- **Релевантность:** $\text{rel}_i$ — оценка релевантности документа. Обычно 0 (нерелевантный), 1 (частично релевантный), 2 (релевантный).
- **Усиление:** $2^{\text{rel}_i} - 1$. Для rel = 0: 0. Для rel = 1: 1. Для rel = 2: 3.
- **Дисконтирование:** $\log_2(i + 1)$. Для позиции 1: $\log_2(2) = 1$. Для позиции 2: $\log_2(3) = 1.58$. Для позиции 10: $\log_2(11) = 3.46$.

**Интуиция:** DCG суммирует релевантность документов, но **дисконтирует** более низкие позиции. Документ на позиции 1 важнее, чем на позиции 10.

**IDCG:** DCG для **идеального** ранжирования (все релевантные документы вверху). nDCG = DCG / IDCG. nDCG@10 = 1, если ранжирование идеальное.

**Тонкий момент:** nDCG@10 — стандартная метрика для retrieval. Она учитывает и релевантность, и порядок. В BEIR и MTEB используется именно nDCG@10.

### 2.7 STS (Semantic Textual Similarity)

**Задача:** даны пары предложений, нужно оценить их семантическую близость.

**Процедура:**

1. Вычислить эмбеддинги для обоих предложений.
2. Вычислить косинусную близость.
3. Вычислить Spearman correlation с человеческими оценками.

**Spearman correlation:**

$$
\rho = 1 - \frac{6 \sum_{i=1}^{n} d_i^2}{n(n^2 - 1)},
$$

где $d_i$ — разность рангов между предсказанной и истинной близостью.

**Разберём формулу:** Spearman correlation измеряет, насколько **ранги** предсказанных близостей совпадают с рангами истинных. Если модель предсказывает близости в том же порядке, что и люди, $\rho \approx 1$.

**Тонкий момент:** STS не требует **абсолютной** точности. Важен только **порядок**: если модель говорит, что пара A ближе, чем пара B, и люди согласны, correlation высокий.

### 2.8 Summarization

**Задача:** оценить качество машинно-сгенерированных суммаризаций.

**Процедура:**

1. Вычислить эмбеддинги для сгенерированной и эталонной суммаризации.
2. Вычислить косинусную близость.
3. Вычислить Spearman correlation с человеческими оценками.

**Тонкий момент:** в MTEB только **один** датасет для summarization. Это означает, что метрика менее надёжна, чем для других задач.

---

## 3. BEIR: Benchmarking IR

### 3.1 Общая идея

**BEIR** — это бенчмарк для **zero-shot retrieval**. Ключевая идея: модель обучается на **одном** датасете (например, MS MARCO), а тестируется на **других** датасетах, которые она **не видела** при обучении.

**Зачем:** в реальных задачах модель часто применяется к **новому** домену (медицина, юриспруденция, наука). Если модель обучалась только на общих данных, сможет ли она искать в новом домене?

**BEIR включает 18 датасетов из разных доменов:**

- **Медицина:** NFCorpus, BioASQ.
- **Наука:** SciFact, SciDocs, SCIDOCS.
- **Новости:** TREC-COVID, Climate-FEVER.
- **Вопросы:** Natural Questions, HotpotQA, FiQA.
- **Факты:** FEVER, DBPedia.
- **Разное:** ArguAna, Touché-2020, Quora, CQADupstack.

**Тонкий момент:** BEIR — это **гетерогенный** бенчмарк. Датасеты различаются по:

- **Домену:** медицина, наука, новости, факты.
- **Типу запроса:** вопросы, ключевые слова, утверждения.
- **Размеру:** от 1K до 5M документов.
- **Длине:** от коротких предложений до длинных статей.

### 3.2 Zero-shot retrieval

**Постановка:** дана модель, обученная на датасете $D_{\text{train}}$. Нужно оценить её на датасете $D_{\text{test}}$, который она **не видела** при обучении.

**Формально:**

$$
\text{Score} = \frac{1}{|D_{\text{test}}|} \sum_{q \in D_{\text{test}}} \text{nDCG@10}(q).
$$

**Тонкий момент:** zero-shot не означает, что модель **вообще** не обучалась. Она обучалась на **другом** датасете. Вопрос в том, насколько хорошо она **обобщает** на новый домен.

**Пример:** модель обучается на MS MARCO (общие вопросы). Затем тестируется на NFCorpus (медицинские вопросы). Если модель хорошо ищет в медицине, она обобщает.

### 3.3 Результаты BEIR

**BM25 как baseline:**

BM25 — это **лексический** метод поиска, основанный на совпадении слов. Он не использует эмбеддинги. BM25 показывает **surprisingly strong** результаты на BEIR:

- **Средний nDCG@10:** ~40.0.

**Dense retrieval:**

Dense-модели (SBERT, E5, BGE, GTE) показывают:

- **SBERT:** ~42.0.
- **E5-large:** ~49.0.
- **BGE-large:** ~52.0.
- **GTE-large:** ~53.0.

**Late interaction:**

ColBERTv2 показывает ~50.0.

**Re-ranking:**

Cross-encoder re-ranking поверх BM25 показывает ~48.0.

**Тонкий момент:** BM25 остаётся **сильным** baseline. Dense-модели превосходят его, но не **катастрофически**. Это означает, что zero-shot retrieval — **сложная** задача.

### 3.4 Почему BEIR важен

**BEIR показывает реальную картину:** модель, которая хорошо работает на MS MARCO, может **плохо** работать на медицинских данных. BEIR измеряет **обобщение**, а не запоминание.

**BEIR используется для:**

1. **Оценки моделей:** сравнение E5, BGE, GTE на zero-shot retrieval.
2. **Выбора модели:** если ваша задача — поиск в новом домене, смотрите на BEIR.
3. **Разработки:** BEIR показывает, где модели слабы, и мотивирует улучшения.

---

## 4. Формулы метрик: полный вывод

### 4.1 Precision@k

$$
P@k = \frac{\text{Number of relevant documents in top-}k}{k}.
$$

**Разберём:** если в top-10 из 10 документов 3 релевантных, $P@10 = 0.3$.

**Тонкий момент:** precision может быть высокой, если документов мало. Например, если в top-1 документ релевантный, $P@1 = 1.0$. Но это не означает, что модель хороша.

### 4.2 Recall@k

$$
R@k = \frac{\text{Number of relevant documents in top-}k}{\text{Total number of relevant documents}}.
$$

**Разберём:** если всего 5 релевантных документов, а в top-10 найдено 3, $R@10 = 0.6$.

**Тонкий момент:** recall может быть высоким, если найти **все** релевантные документы, но они могут быть в **конце** списка. Recall не учитывает порядок.

### 4.3 Average Precision (AP)

$$
\text{AP} = \frac{1}{R} \sum_{k=1}^{n} P@k \cdot \text{rel}(k),
$$

где $R$ — общее число релевантных документов, $\text{rel}(k)$ — индикатор релевантности на позиции $k$.

**Разберём:** AP — это среднее precision на **каждой** позиции, где есть релевантный документ.

**Пример:** релевантные документы на позициях 1, 3, 5. Тогда:

$$
\text{AP} = \frac{1}{3} (P@1 + P@3 + P@5) = \frac{1}{3} (1.0 + 0.67 + 0.6) = 0.756.
$$

### 4.4 Mean Average Precision (MAP)

$$
\text{MAP} = \frac{1}{Q} \sum_{q=1}^{Q} \text{AP}(q).
$$

**Разберём:** MAP — это среднее AP по всем запросам. MAP = 1, если для каждого запроса все релевантные документы вверху.

### 4.5 Mean Reciprocal Rank (MRR)

$$
\text{MRR} = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{\text{rank}_q},
$$

где $\text{rank}_q$ — позиция первого релевантного документа.

**Разберём:** MRR фокусируется на **первом** релевантном документе. Если он на позиции 1, вклад = 1. Если на позиции 3, вклад = 1/3 = 0.33.

### 4.6 Discounted Cumulative Gain (DCG)

$$
\text{DCG@k} = \sum_{i=1}^{k} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}.
$$

**Разберём по частям:**

- $2^{\text{rel}_i} - 1$: усиление. Для rel = 0: 0. Для rel = 1: 1. Для rel = 2: 3.
- $\log_2(i + 1)$: дисконтирование. Для позиции 1: 1. Для позиции 2: 1.58. Для позиции 10: 3.46.

**Пример:** документы с релевантностью 2, 1, 0 на позициях 1, 2, 3.

$$
\text{DCG@3} = \frac{2^2 - 1}{\log_2(2)} + \frac{2^1 - 1}{\log_2(3)} + \frac{2^0 - 1}{\log_2(4)} = \frac{3}{1} + \frac{1}{1.58} + \frac{0}{2} = 3 + 0.63 + 0 = 3.63.
$$

### 4.7 Normalized DCG (nDCG)

$$
\text{nDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}},
$$

где IDCG@k — DCG для **идеального** ранжирования.

**Пример:** если идеальное ранжирование даёт DCG = 4.0, а модель даёт 3.63, nDCG = 0.91.

**Тонкий момент:** nDCG@10 — стандартная метрика для retrieval. Она учитывает и релевантность, и порядок. nDCG@10 = 1, если ранжирование идеальное.

---

## 5. MTEB vs BEIR: сравнение

| Свойство | MTEB | BEIR |
|----------|------|------|
| **Тип** | Широкий бенчмарк | Узкий бенчмарк |
| **Задачи** | 7+ типов | Только retrieval |
| **Датасеты** | 56+ | 18 |
| **Языки** | 112+ | В основном английский |
| **Метрики** | nDCG, MAP, V-measure, F1, Spearman | nDCG@10 |
| **Zero-shot** | Да (для некоторых задач) | Да (основная идея) |
| **Лидерборд** | HuggingFace MTEB | BEIR leaderboard |
| **Использование** | Общая оценка моделей | Оценка zero-shot retrieval |

**Тонкий момент:** MTEB включает **retrieval** как одну из задач, используя **BEIR** датасеты. То есть BEIR — это **подмножество** MTEB. Но BEIR фокусируется **только** на retrieval, что позволяет глубже анализировать эту задачу.

---

## 6. Лидерборды и современные результаты

### 6.1 MTEB Leaderboard

**MTEB Leaderboard** — это публичный лидерборд на HuggingFace, где модели сравниваются по среднему баллу на MTEB.

**Структура:**

- **Модель:** название.
- **Размер:** число параметров.
- **Embedding size:** размерность эмбеддинга.
- **Max tokens:** максимальная длина.
- **Средний балл:** среднее по всем задачам.
- **Баллы по задачам:** classification, clustering, retrieval, STS, etc.

**Топовые модели (2026):**

| Модель | Средний балл | Retrieval | STS |
|--------|--------------|-----------|-----|
| Qwen3-Embedding-8B | 70.58 | — | — |
| Octen-Embedding-8B | 0.8045 (mean task) | — | — |
| Harrier | — | — | — |
| GTE-large | 65.0 | 53.0 | 84.0 |
| BGE-large | 64.0 | 52.0 | 83.0 |
| E5-large | 62.0 | 50.0 | 82.0 |

**Тонкий момент:** лидерборд **динамичен**. Новые модели появляются каждые несколько месяцев. В 2026 году топовые модели — Qwen3-Embedding-8B, Octen-Embedding-8B, Harrier.

### 6.2 BEIR Leaderboard

**BEIR Leaderboard** — это лидерборд для zero-shot retrieval. Модели сравниваются по среднему nDCG@10 на 18 датасетах.

**Топовые модели:**

| Модель | Средний nDCG@10 |
|--------|-----------------|
| GTE-large | 53.0 |
| BGE-large | 52.0 |
| ColBERTv2 | 50.0 |
| E5-large | 49.0 |
| SBERT | 42.0 |
| BM25 | 40.0 |

**Тонкий момент:** BM25 остаётся **сильным** baseline. Dense-модели превосходят его, но не **катастрофически**. Это означает, что zero-shot retrieval — **сложная** задача.

---

## 7. Практические аспекты

### 7.1 Как выбрать модель

**Если ваша задача — retrieval:**

- Смотрите на **BEIR** и **MTEB retrieval**.
- Топ: GTE-large, BGE-large, E5-large.
- Для русского: multilingual-e5-large, bge-m3.

**Если ваша задача — STS:**

- Смотрите на **MTEB STS**.
- Топ: GTE-large, BGE-large, E5-large.

**Если ваша задача — classification:**

- Смотрите на **MTEB classification**.
- Топ: Instructor-xl, BGE-large.

**Если ваша задача — clustering:**

- Смотрите на **MTEB clustering**.
- Топ: GTE-large, BGE-large.

**Тонкий момент:** не всегда «лучшая» модель на MTEB — лучшая для **вашей** задачи. Проверяйте на своих данных.

### 7.2 Как оценить модель на своих данных

**Шаги:**

1. **Собрать данные:** пары (query, document) с метками релевантности.
2. **Разделить:** train/test.
3. **Обучить модель** (если нужно) на train.
4. **Оценить** на test: nDCG@10, MRR@10, Recall@10.
5. **Сравнить** с baseline (BM25).

**Тонкий момент:** если у вас **мало** данных, используйте **zero-shot** оценку: обучите модель на другом датасете, протестируйте на вашем.

### 7.3 Как использовать MTEB и BEIR

**MTEB:**

```python
import mteb
from mteb import MTEB

model = ...
evaluation = MTEB(tasks=["MSMARCO", "SciFact"])
results = evaluation.run(model)
```

**BEIR:**

```python
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

corpus, queries, qrels = GenericDataLoader("scifact").load(split="test")
retriever = ...
results = retriever.retrieve(corpus, queries)
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, [10])
```

**Тонкий момент:** MTEB и BEIR — это **библиотеки**, а не только бенчмарки. Вы можете использовать их для оценки **своих** моделей.

### 7.4 Ограничения бенчмарков

**Первое ограничение: overfitting.** Модели могут **переобучаться** на MTEB, если обучаются на его данных. MTEB исключает модели, которые обучались на >25% данных.

**Второе ограничение: домен.** MTEB включает много доменов, но не **все**. Ваш домен может быть не представлен.

**Третье ограничение: язык.** MTEB включает 112 языков, но не **все**. Для редких языков может не быть данных.

**Четвёртое ограничение: метрики.** Метрики не всегда **коррелируют** с реальным качеством. Модель может быть хороша в nDCG@10, но плоха в **реальном** использовании.

**Тонкий момент:** бенчмарки — это **инструмент**, а не **истина**. Используйте их как **ориентир**, но проверяйте на своих данных.

---

## 8. Заключение

MTEB и BEIR — это два главных бенчмарка для оценки эмбеддингов.

**MTEB:**

- Широкий: 56+ датасетов, 7+ задач, 112+ языков.
- Оценивает classification, clustering, retrieval, STS, etc.
- Лидерборд на HuggingFace.
- Используется для **общей** оценки моделей.

**BEIR:**

- Узкий: 18 датасетов, только retrieval.
- Оценивает **zero-shot** retrieval: обобщение на новый домен.
- Лидерборд на GitHub.
- Используется для **поиска**.

**Ключевые метрики:**

- **nDCG@10:** для retrieval. Учитывает релевантность и порядок.
- **MAP:** для reranking. Среднее precision.
- **MRR:** для reranking. Позиция первого релевантного.
- **V-measure:** для clustering. Homogeneity + completeness.
- **Spearman:** для STS. Correlation с человеческими оценками.
- **F1:** для classification. Баланс precision и recall.

**Формулы:**

$$
\text{nDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}, \quad \text{DCG@k} = \sum_{i=1}^{k} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}.
$$

$$
\text{MAP} = \frac{1}{Q} \sum_{q=1}^{Q} \text{AP}(q).
$$

$$
\text{MRR} = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{\text{rank}_q}.
$$

$$
V = \frac{2 \cdot H \cdot C}{H + C}.
$$

**Практические рекомендации:**

1. Смотрите на **retrieval** метрики для поиска.
2. Смотрите на **STS** метрики для similarity.
3. Смотрите на **classification** метрики для классификации.
4. Проверяйте на **своих** данных.
5. Используйте **BM25** как baseline.
6. Для zero-shot retrieval смотрите на **BEIR**.


# Численный пример метрик MTEB и BEIR: полный пошаговый разбор

## Короткий ответ

**Да, пример нужен.** MTEB и BEIR — это бенчмарки, основанные на **метриках**. Без конкретных вычислений формулы nDCG, MAP, MRR, V-measure, Spearman остаются абстрактными. Пример покажет, как каждая метрика работает на числах.

Ниже — полный разбор **всех** основных метрик MTEB и BEIR на конкретных примерах.

---

## 1. Retrieval: nDCG@k

### 1.1 Постановка

**Запрос:** «как приготовить пасту».

**Корпус:** 5 документов:

- $d_1$: «рецепт спагетти» (релевантность 2)
- $d_2$: «рецепт пиццы» (релевантность 1)
- $d_3$: «прогноз погоды» (релевантность 0)
- $d_4$: «итальянская кухня» (релевантность 1)
- $d_5$: «как сварить макароны» (релевантность 2)

**Модель ранжировала документы так:**

| Позиция | Документ | Релевантность |
|---------|----------|---------------|
| 1 | $d_3$ | 0 |
| 2 | $d_1$ | 2 |
| 3 | $d_5$ | 2 |
| 4 | $d_2$ | 1 |
| 5 | $d_4$ | 1 |

**Задача:** вычислить nDCG@5.

### 1.2 Шаг 1: DCG@5

$$
\text{DCG@5} = \sum_{i=1}^{5} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}.
$$

**Позиция 1:** $\text{rel}_1 = 0$.

$$
\frac{2^0 - 1}{\log_2(2)} = \frac{0}{1} = 0.
$$

**Позиция 2:** $\text{rel}_2 = 2$.

$$
\frac{2^2 - 1}{\log_2(3)} = \frac{3}{1.58496} = 1.8928.
$$

**Позиция 3:** $\text{rel}_3 = 2$.

$$
\frac{2^2 - 1}{\log_2(4)} = \frac{3}{2} = 1.5000.
$$

**Позиция 4:** $\text{rel}_4 = 1$.

$$
\frac{2^1 - 1}{\log_2(5)} = \frac{1}{2.32193} = 0.4307.
$$

**Позиция 5:** $\text{rel}_5 = 1$.

$$
\frac{2^1 - 1}{\log_2(6)} = \frac{1}{2.58496} = 0.3869.
$$

**Сумма:**

$$
\text{DCG@5} = 0 + 1.8928 + 1.5000 + 0.4307 + 0.3869 = 4.2104.
$$

### 1.3 Шаг 2: IDCG@5

**Идеальное ранжирование:** сначала все документы с релевантностью 2, потом с 1, потом с 0.

Идеальный порядок: $d_1$ (2), $d_5$ (2), $d_2$ (1), $d_4$ (1), $d_3$ (0).

**Позиция 1:** $\text{rel} = 2$.

$$
\frac{2^2 - 1}{\log_2(2)} = \frac{3}{1} = 3.0000.
$$

**Позиция 2:** $\text{rel} = 2$.

$$
\frac{3}{\log_2(3)} = \frac{3}{1.58496} = 1.8928.
$$

**Позиция 3:** $\text{rel} = 1$.

$$
\frac{1}{\log_2(4)} = \frac{1}{2} = 0.5000.
$$

**Позиция 4:** $\text{rel} = 1$.

$$
\frac{1}{\log_2(5)} = \frac{1}{2.32193} = 0.4307.
$$

**Позиция 5:** $\text{rel} = 0$.

$$
\frac{0}{\log_2(6)} = 0.
$$

**Сумма:**

$$
\text{IDCG@5} = 3.0000 + 1.8928 + 0.5000 + 0.4307 + 0 = 5.8235.
$$

### 1.4 Шаг 3: nDCG@5

$$
\text{nDCG@5} = \frac{\text{DCG@5}}{\text{IDCG@5}} = \frac{4.2104}{5.8235} = 0.7230.
$$

**Интерпретация:** nDCG@5 = 0.72. Это означает, что модель ранжировала документы на 72% так же хорошо, как идеальное ранжирование.

**Тонкий момент:** если бы модель поставила $d_1$ и $d_5$ на позиции 1 и 2, nDCG был бы 1.0. Если бы все релевантные документы были внизу, nDCG был бы близок к 0.

---

## 2. Reranking: MAP и MRR

### 2.1 Постановка

**Два запроса:**

- $q_1$: «как приготовить пасту»
- $q_2$: «прогноз погоды»

**Результаты для $q_1$:**

| Позиция | Релевантность |
|---------|---------------|
| 1 | 1 |
| 2 | 0 |
| 3 | 1 |
| 4 | 0 |
| 5 | 1 |

**Результаты для $q_2$:**

| Позиция | Релевантность |
|---------|---------------|
| 1 | 0 |
| 2 | 1 |
| 3 | 0 |
| 4 | 0 |
| 5 | 0 |

**Задача:** вычислить MAP и MRR.

### 2.2 Average Precision для $q_1$

**Релевантные на позициях 1, 3, 5.**

$$
\text{AP}(q_1) = \frac{1}{3} \left( P@1 + P@3 + P@5 \right).
$$

**$P@1$:** на позиции 1 релевантный.

$$
P@1 = \frac{1}{1} = 1.0.
$$

**$P@3$:** на позициях 1–3 два релевантных (1 и 3).

$$
P@3 = \frac{2}{3} = 0.6667.
$$

**$P@5$:** на позициях 1–5 три релевантных (1, 3, 5).

$$
P@5 = \frac{3}{5} = 0.6.
$$

$$
\text{AP}(q_1) = \frac{1}{3} (1.0 + 0.6667 + 0.6) = \frac{2.2667}{3} = 0.7556.
$$

### 2.3 Average Precision для $q_2$

**Релевантный на позиции 2.**

$$
\text{AP}(q_2) = \frac{1}{1} \cdot P@2 = \frac{1}{2} = 0.5.
$$

### 2.4 MAP

$$
\text{MAP} = \frac{1}{2} (\text{AP}(q_1) + \text{AP}(q_2)) = \frac{1}{2} (0.7556 + 0.5) = 0.6278.
$$

### 2.5 MRR

**Для $q_1$:** первый релевантный на позиции 1.

$$
\frac{1}{\text{rank}_{q_1}} = \frac{1}{1} = 1.0.
$$

**Для $q_2$:** первый релевантный на позиции 2.

$$
\frac{1}{\text{rank}_{q_2}} = \frac{1}{2} = 0.5.
$$

$$
\text{MRR} = \frac{1}{2} (1.0 + 0.5) = 0.75.
$$

**Интерпретация:** MAP = 0.63, MRR = 0.75. MRR выше, потому что первый релевантный документ для $q_1$ на позиции 1, для $q_2$ — на позиции 2. MAP учитывает **все** релевантные документы, поэтому ниже.

---

## 3. Classification: Precision, Recall, F1

### 3.1 Постановка

**Классификация тональности:** 10 отзывов, метки {positive, negative}.

**Истинные метки:**

| № | Истина | Предсказание |
|---|--------|--------------|
| 1 | positive | positive |
| 2 | positive | positive |
| 3 | negative | positive |
| 4 | negative | negative |
| 5 | positive | positive |
| 6 | negative | negative |
| 7 | positive | negative |
| 8 | negative | negative |
| 9 | positive | positive |
| 10 | negative | negative |

**Задача:** вычислить Precision, Recall, F1 для класса «positive».

### 3.2 Confusion matrix

**True Positive (TP):** истина positive, предсказание positive.

- № 1, 2, 5, 9 → TP = 4.

**False Positive (FP):** истина negative, предсказание positive.

- № 3 → FP = 1.

**False Negative (FN):** истина positive, предсказание negative.

- № 7 → FN = 1.

**True Negative (TN):** истина negative, предсказание negative.

- № 4, 6, 8, 10 → TN = 4.

### 3.3 Precision

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} = \frac{4}{4 + 1} = \frac{4}{5} = 0.8.
$$

**Интерпретация:** из всех предсказанных «positive» (5 штук) 4 действительно positive. Точность 80%.

### 3.4 Recall

$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{4}{4 + 1} = \frac{4}{5} = 0.8.
$$

**Интерпретация:** из всех истинных «positive» (5 штук) 4 найдены. Полнота 80%.

### 3.5 F1

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = 2 \cdot \frac{0.8 \cdot 0.8}{0.8 + 0.8} = 2 \cdot \frac{0.64}{1.6} = 0.8.
$$

**Интерпретация:** F1 = 0.8. Это баланс между precision и recall.

**Тонкий момент:** если precision = 1.0, а recall = 0.1, F1 = 0.18. F1 **штрафует** модели, которые хороши только в одном из двух.

---

## 4. Clustering: V-measure

### 4.1 Постановка

**10 документов** с истинными классами (A, B, C) и предсказанными кластерами (1, 2).

| № | Истинный класс | Кластер |
|---|----------------|---------|
| 1 | A | 1 |
| 2 | A | 1 |
| 3 | A | 1 |
| 4 | B | 2 |
| 5 | B | 2 |
| 6 | B | 1 |
| 7 | C | 1 |
| 8 | C | 2 |
| 9 | C | 2 |
| 10 | C | 2 |

**Задача:** вычислить homogeneity, completeness, V-measure.

### 4.2 Homogeneity

**Homogeneity** = 1, если каждый кластер содержит только один класс.

**Кластер 1:** документы 1, 2, 3 (A), 6 (B), 7 (C). Классы: A, B, C → **не однородный**.

**Кластер 2:** документы 4, 5 (B), 8, 9, 10 (C). Классы: B, C → **не однородный**.

**Вычислим энтропии.**

**Энтропия классов $H(C)$:**

Классы: A (3 документа), B (3), C (4).

$$
H(C) = -\frac{3}{10} \log_2 \frac{3}{10} - \frac{3}{10} \log_2 \frac{3}{10} - \frac{4}{10} \log_2 \frac{4}{10}.
$$

$$
= -0.3 \cdot (-1.7370) - 0.3 \cdot (-1.7370) - 0.4 \cdot (-1.3219).
$$

$$
= 0.5211 + 0.5211 + 0.5288 = 1.5710.
$$

**Условная энтропия $H(C \mid K)$:**

Для каждого кластера вычислим энтропию классов, взвесим по размеру.

**Кластер 1 (5 документов):** A — 3, B — 1, C — 1.

$$
H(C \mid K=1) = -\frac{3}{5} \log_2 \frac{3}{5} - \frac{1}{5} \log_2 \frac{1}{5} - \frac{1}{5} \log_2 \frac{1}{5}.
$$

$$
= -0.6 \cdot (-0.7370) - 0.2 \cdot (-2.3219) - 0.2 \cdot (-2.3219).
$$

$$
= 0.4422 + 0.4644 + 0.4644 = 1.3710.
$$

**Кластер 2 (5 документов):** B — 2, C — 3.

$$
H(C \mid K=2) = -\frac{2}{5} \log_2 \frac{2}{5} - \frac{3}{5} \log_2 \frac{3}{5}.
$$

$$
= -0.4 \cdot (-1.3219) - 0.6 \cdot (-0.7370) = 0.5288 + 0.4422 = 0.9710.
$$

**Взвешенная сумма:**

$$
H(C \mid K) = \frac{5}{10} \cdot 1.3710 + \frac{5}{10} \cdot 0.9710 = 0.6855 + 0.4855 = 1.1710.
$$

**Homogeneity:**

$$
H_{\text{hom}} = 1 - \frac{H(C \mid K)}{H(C)} = 1 - \frac{1.1710}{1.5710} = 1 - 0.7454 = 0.2546.
$$

### 4.3 Completeness

**Completeness** = 1, если все документы одного класса в одном кластере.

**Класс A:** документы 1, 2, 3 — все в кластере 1 → **полный**.

**Класс B:** документы 4, 5 в кластере 2, документ 6 в кластере 1 → **не полный**.

**Класс C:** документы 7 в кластере 1, 8, 9, 10 в кластере 2 → **не полный**.

**Энтропия кластеров $H(K)$:**

Кластер 1: 5 документов, кластер 2: 5 документов.

$$
H(K) = -\frac{5}{10} \log_2 \frac{5}{10} - \frac{5}{10} \log_2 \frac{5}{10} = -0.5 \cdot (-1) - 0.5 \cdot (-1) = 1.0.
$$

**Условная энтропия $H(K \mid C)$:**

**Класс A (3 документа):** все в кластере 1.

$$
H(K \mid C=A) = 0.
$$

**Класс B (3 документа):** 2 в кластере 2, 1 в кластере 1.

$$
H(K \mid C=B) = -\frac{2}{3} \log_2 \frac{2}{3} - \frac{1}{3} \log_2 \frac{1}{3}.
$$

$$
= -0.6667 \cdot (-0.5850) - 0.3333 \cdot (-1.5850) = 0.3900 + 0.5283 = 0.9183.
$$

**Класс C (4 документа):** 1 в кластере 1, 3 в кластере 2.

$$
H(K \mid C=C) = -\frac{1}{4} \log_2 \frac{1}{4} - \frac{3}{4} \log_2 \frac{3}{4}.
$$

$$
= -0.25 \cdot (-2) - 0.75 \cdot (-0.4150) = 0.5 + 0.3113 = 0.8113.
$$

**Взвешенная сумма:**

$$
H(K \mid C) = \frac{3}{10} \cdot 0 + \frac{3}{10} \cdot 0.9183 + \frac{4}{10} \cdot 0.8113 = 0 + 0.2755 + 0.3245 = 0.6000.
$$

**Completeness:**

$$
H_{\text{comp}} = 1 - \frac{H(K \mid C)}{H(K)} = 1 - \frac{0.6000}{1.0} = 1 - 0.6 = 0.4.
$$

### 4.4 V-measure

$$
V = \frac{2 \cdot H_{\text{hom}} \cdot H_{\text{comp}}}{H_{\text{hom}} + H_{\text{comp}}} = \frac{2 \cdot 0.2546 \cdot 0.4}{0.2546 + 0.4} = \frac{0.2037}{0.6546} = 0.3112.
$$

**Интерпретация:** V-measure = 0.31. Это низкое значение, потому что кластеры **не совпадают** с классами: кластер 1 содержит A, B, C; кластер 2 содержит B, C.

**Тонкий момент:** идеальная кластеризация дала бы V = 1.0. V = 0.31 означает, что кластеризация **слабая**.

---

## 5. STS: Spearman correlation

### 5.1 Постановка

**5 пар предложений** с истинными оценками близости (от 0 до 5):

| Пара | Истинная близость |
|------|-------------------|
| 1 | 5.0 |
| 2 | 4.0 |
| 3 | 3.0 |
| 4 | 2.0 |
| 5 | 1.0 |

**Модель предсказала близости:**

| Пара | Предсказанная близость |
|------|------------------------|
| 1 | 0.95 |
| 2 | 0.80 |
| 3 | 0.60 |
| 4 | 0.40 |
| 5 | 0.20 |

**Задача:** вычислить Spearman correlation.

### 5.2 Ранги

**Истинные ранги:** 5, 4, 3, 2, 1.

**Предсказанные ранги:** 5, 4, 3, 2, 1 (потому что порядок совпадает).

**Разности рангов $d_i$:**

$$
d_1 = 5 - 5 = 0, \quad d_2 = 4 - 4 = 0, \quad d_3 = 3 - 3 = 0, \quad d_4 = 2 - 2 = 0, \quad d_5 = 1 - 1 = 0.
$$

### 5.3 Spearman

$$
\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)} = 1 - \frac{6 \cdot 0}{5 \cdot 24} = 1 - 0 = 1.0.
$$

**Интерпретация:** Spearman = 1.0. Модель идеально ранжирует пары.

### 5.4 Что если модель ошибается

Пусть модель предсказала:

| Пара | Предсказанная близость | Ранг |
|------|------------------------|------|
| 1 | 0.20 | 1 |
| 2 | 0.40 | 2 |
| 3 | 0.60 | 3 |
| 4 | 0.80 | 4 |
| 5 | 0.95 | 5 |

**Истинные ранги:** 5, 4, 3, 2, 1.

**Предсказанные ранги:** 1, 2, 3, 4, 5.

**Разности:**

$$
d_1 = 5 - 1 = 4, \quad d_2 = 4 - 2 = 2, \quad d_3 = 3 - 3 = 0, \quad d_4 = 2 - 4 = -2, \quad d_5 = 1 - 5 = -4.
$$

**Сумма квадратов:**

$$
\sum d_i^2 = 16 + 4 + 0 + 4 + 16 = 40.
$$

$$
\rho = 1 - \frac{6 \cdot 40}{5 \cdot 24} = 1 - \frac{240}{120} = 1 - 2 = -1.0.
$$

**Интерпретация:** Spearman = −1.0. Модель ранжирует пары **в обратном порядке**. Это худший случай.

**Тонкий момент:** Spearman correlation **не зависит** от абсолютных значений близости. Важен только **порядок**. Если модель говорит, что пара A ближе, чем пара B, и люди согласны, correlation высокий.

---

## 6. Bitext Mining: F1

### 6.1 Постановка

**Задача:** найти параллельные предложения в двух языках.

**Английские предложения:**

- $e_1$: «The cat sits on the window»
- $e_2$: «The dog sleeps on the sofa»
- $e_3$: «The weather is nice today»

**Русские предложения:**

- $r_1$: «Кошка сидит на окне»
- $r_2$: «Собака спит на диване»
- $r_3$: «Погода сегодня хорошая»

**Правильные пары:** $(e_1, r_1)$, $(e_2, r_2)$, $(e_3, r_3)$.

**Модель предсказала:** $(e_1, r_1)$, $(e_2, r_3)$, $(e_3, r_2)$.

### 6.2 Confusion matrix

**True Positive (TP):** правильные пары, которые модель нашла.

- $(e_1, r_1)$ → TP = 1.

**False Positive (FP):** неправильные пары, которые модель предсказала.

- $(e_2, r_3)$, $(e_3, r_2)$ → FP = 2.

**False Negative (FN):** правильные пары, которые модель **не** нашла.

- $(e_2, r_2)$, $(e_3, r_3)$ → FN = 2.

### 6.3 Precision, Recall, F1

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} = \frac{1}{1 + 2} = \frac{1}{3} = 0.3333.
$$

$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{1}{1 + 2} = \frac{1}{3} = 0.3333.
$$

$$
F_1 = 2 \cdot \frac{0.3333 \cdot 0.3333}{0.3333 + 0.3333} = 2 \cdot \frac{0.1111}{0.6667} = 0.3333.
$$

**Интерпретация:** F1 = 0.33. Модель нашла только 1 из 3 правильных пар.

---

## 7. Summarization: Spearman correlation

### 7.1 Постановка

**5 суммаризаций** с человеческими оценками (от 1 до 5) и оценками модели (косинусная близость):

| Суммаризация | Человеческая оценка | Оценка модели |
|--------------|---------------------|---------------|
| 1 | 5 | 0.85 |
| 2 | 4 | 0.75 |
| 3 | 3 | 0.65 |
| 4 | 2 | 0.55 |
| 5 | 1 | 0.45 |

**Задача:** вычислить Spearman correlation.

### 7.2 Ранги

**Человеческие ранги:** 5, 4, 3, 2, 1.

**Ранги модели:** 5, 4, 3, 2, 1.

**Разности:** все 0.

$$
\rho = 1.0.
$$

**Интерпретация:** модель идеально ранжирует суммаризации.

---

## 8. Сводка всех метрик

| Метрика | Задача | Формула | Пример |
|---------|--------|---------|--------|
| nDCG@k | Retrieval | $\frac{\text{DCG}}{\text{IDCG}}$ | 0.72 |
| MAP | Reranking | $\frac{1}{Q} \sum \text{AP}$ | 0.63 |
| MRR | Reranking | $\frac{1}{Q} \sum \frac{1}{\text{rank}}$ | 0.75 |
| Precision | Classification | $\frac{\text{TP}}{\text{TP} + \text{FP}}$ | 0.80 |
| Recall | Classification | $\frac{\text{TP}}{\text{TP} + \text{FN}}$ | 0.80 |
| F1 | Classification | $2 \cdot \frac{P \cdot R}{P + R}$ | 0.80 |
| V-measure | Clustering | $\frac{2 H C}{H + C}$ | 0.31 |
| Spearman | STS | $1 - \frac{6 \sum d^2}{n(n^2-1)}$ | 1.0 |

---

## 9. Заключение

В этом численном примере мы шаг за шагом вычислили **все** основные метрики MTEB и BEIR:

1. **nDCG@k** — для retrieval. Учитывает релевантность и порядок. Пример: 0.72.

2. **MAP** — для reranking. Среднее precision по всем запросам. Пример: 0.63.

3. **MRR** — для reranking. Позиция первого релевантного. Пример: 0.75.

4. **Precision, Recall, F1** — для classification. Пример: 0.80.

5. **V-measure** — для clustering. Homogeneity + completeness. Пример: 0.31.

6. **Spearman** — для STS. Correlation с человеческими оценками. Пример: 1.0.

7. **F1 для bitext mining** — для поиска параллельных предложений. Пример: 0.33.

**Ключевые формулы:**

nDCG:
$$
\text{nDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}, \quad \text{DCG@k} = \sum_{i=1}^{k} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}.
$$

MAP:
$$
\text{MAP} = \frac{1}{Q} \sum_{q=1}^{Q} \text{AP}(q).
$$

MRR:
$$
\text{MRR} = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{\text{rank}_q}.
$$

F1:
$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}.
$$

V-measure:
$$
V = \frac{2 \cdot H \cdot C}{H + C}.
$$

Spearman:
$$
\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}.
$$

Этот пример показывает, как **интерпретировать** метрики MTEB и BEIR. В реальных задачах вы будете вычислять эти метрики с помощью библиотек (MTEB, BEIR), но понимание формул необходимо для **правильной** интерпретации результатов.
